<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L06-proteogenomics/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L06-proteogenomics/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/proteomics-multiomics/lessons/P07-L06-proteogenomics/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/proteomics-multiomics/lessons/P07-L06-proteogenomics/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P07-L06 · Proteogenomics: from a variant to protein evidence, and how many variant peptides survive the spectra

**You will build:** the path from a called single-nucleotide variant to a claim about protein. You will translate a
gene and say what a variant does to it, list the peptides that exist only because of the variant, add them to a
search database with decoys that belong to them, give the variant peptides an error rate of their own, check each
variant spectrum against the reference peptide it could really be, and count the peptides that stand behind each
variant. Then you will **measure**, against an answer key, how many reported variants have spectra behind them.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download · **Prerequisites:**
`P07-L01-search-engine-and-fdr` (the search engine and target-decoy counting, carried here unchanged) and
`P07-L05-leakage-harness` (the habit of asking a check what it cannot see). Python and numpy.

By the end you will be able to:

1. Implement translation of a coding sequence and the effect of a single-nucleotide variant on it, from a
   genome coordinate and a strand, and say which variants cannot change a protein.
2. Implement the list of variant-specific peptides, treating leucine and isoleucine as one residue because no mass
   spectrometer separates them, and a search database whose decoys keep the group of the peptide they came from.
3. Implement a target-decoy error rate computed inside a group, and measure how much it differs from the pooled one
   for variant peptides.
4. Implement a precursor-mass check of a variant spectrum against its reference twin, and a count of the distinct
   peptides behind each variant.
5. Measure, over a set of simulated genomes, how many reported variants have multi-peptide, single-peptide or no
   spectral support, and explain which errors a false discovery rate cannot see.

**The data is synthetic, and the simulator is in this notebook.** A real cohort has no answer key, so it can never
tell you which of its reported variants were ever present in a spectrum. A simulated one does. `meta.yaml` declares
the data as synthetic and nothing is downloaded. The catalogue's version of this lab maps variants onto a public
proteogenomic cohort; this lesson does not, and nothing you conclude here describes a real cohort.

**What this lesson does not do.** No introns, no splicing, no transcript-level evidence, no heterozygous haplotypes
(each variant is applied alone), no modified peptides, no insertions or deletions, and one toy search engine. The sizes
printed below are what these worlds show.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `P07-L01`, `P07-L05`

**This lesson assumes you can already:**

- write Python functions over strings, dicts and sets, and slice a string by position
- read a DNA sequence as codons and say what the reverse complement of a strand is
- describe target-decoy error control, as P07-L01 builds it: decoy hits stand in for wrong target hits, and the false discovery rate is estimated as decoys over targets
- index a numpy array with a boolean mask and use np.unique to get one entry per label

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Proteogenomics: concepts, applications and computational strategies](https://pmc.ncbi.nlm.nih.gov/articles/PMC4392723/) · Alexey I. Nesvizhskii (Nature Methods 2014, PubMed Central) · *free* — what a customised variant database is, how it is built from genomic and transcriptomic data, and why false positive identifications are its main challenge, which is the whole of this lesson
- [Personalized Proteome: Comparing Proteogenomics and Open Variant Search Approaches for Single Amino Acid Variant Detection](https://pmc.ncbi.nlm.nih.gov/articles/PMC8280751/) · Salz, Bouwmeester, Gabriels, Degroeve, Martens, Volders and 't Hoen (Journal of Proteome Research 2021, PubMed Central) · *free* — the methods section on separate error control for the variant peptide subset and on why it needs enough variant decoys, which is exercise 5
- [The Genetic Codes](https://www.ncbi.nlm.nih.gov/Taxonomy/Utils/wprintgc.cgi) · NCBI Taxonomy · *free* — translation table 1, the standard code that CODON_TABLE is built from, for exercises 1 and 2
- [Mascot: Amino acid reference data](https://www.matrixscience.com/help/aa_help.html) · Matrix Science · *free* — the residue masses; compare leucine with isoleucine and lysine with glutamine to see the two gaps that exercises 3 and 6 turn on
- [Improved False Discovery Rate Estimation Procedure for Shotgun Proteomics](https://pmc.ncbi.nlm.nih.gov/articles/PMC4533616/) · Keich, Kertesz-Farkas and Noble (Journal of Proteome Research 2015, PubMed Central) · *free* — the target-decoy estimate of the false discovery rate as decoys over targets, which every error rate in the lesson is built from

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import math
import sys
import traceback
from typing import Callable, NamedTuple

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

# The world. Every number that shapes it is here, so that nothing later is a hidden knob.
SEED = 20261112              # this lesson's one seed: every world derives from it, so re-running prints the same
                             # numbers anywhere
N_WORLDS = 6                 # independently generated worlds; every table below averages over them
N_GENES = 400                # genes per world, each on one strand of one chromosome
N_VARIANTS = 2400            # candidate single-nucleotide variants called in the cohort (most are never seen at protein level)
P_GENE = 0.15                # chance that a gene is expressed in the sample
P_EXPRESS = 0.04             # chance that a variant allele is expressed (which forces its gene to be expressed)
P_DETECT = 0.5               # chance that a peptide of an expressed protein is selected for fragmentation
FOREIGN_SHARE = 0.45         # share of all spectra that come from peptides in no database (modified, novel, contaminant)

# The search settings, carried over from P07-L01 and fixed for the whole lesson so that only the error control varies.
MISSED = 1                   # missed cleavages allowed
MIN_LEN, MAX_LEN = 7, 30
PRECURSOR_PPM = 20.0         # precursor mass tolerance, parts per million of the observed mass
FRAGMENT_TOL = 0.5           # fragment m/z tolerance in Da: a low-resolution instrument
ALPHA = 0.01                 # the one-percent line, set here rather than typed in prose
MIN_DECOYS = 10              # a group with fewer decoy hits than this has no usable error estimate (exercise 5)

# Monoisotopic residue masses (Matrix Science, see claims.yaml) and the standard genetic code (NCBI table 1).
PROTON = 1.0072764666        # mass of a proton, in daltons
WATER = 18.0105646838        # one water: the two ends of an intact peptide
RESIDUE_MASS = {
    "G": 57.021464, "A": 71.037114, "S": 87.032028, "P": 97.052764, "V": 99.068414,
    "T": 101.047679, "C": 103.009185, "L": 113.084064, "I": 113.084064, "N": 114.042927,
    "D": 115.026943, "Q": 128.058578, "K": 128.094963, "E": 129.042593, "M": 131.040485,
    "H": 137.058912, "F": 147.068414, "R": 156.101111, "Y": 163.06332, "W": 186.079313,
}
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
_CODE = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"   # NCBI table 1, bases ordered T, C, A, G
CODON_TABLE = {a + b + c: _CODE[16 * i + 4 * j + k]
               for i, a in enumerate("TCAG") for j, b in enumerate("TCAG") for k, c in enumerate("TCAG")}
COMPLEMENT = {"A": "T", "C": "G", "G": "C", "T": "A"}

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
_ROW_FAILS: dict[str, dict[str, str]] = {}   # exercise -> {table row: what disagreed}, from the latest run of each table row

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("translate",),
    "exercise 2": ("snv_effect",),
    "exercise 3": ("novel_peptides",),
    "exercise 4": ("build_space",),
    "exercise 5": ("group_q_values",),
    "exercise 6": ("twin_verdict",),
    "exercise 7": ("variant_support",),
}
# The demos (the tables below the exercises) run when this file is the program being run, that is in a
# notebook or as `python lesson.py`; the autograder imports the file and only needs the checks.
_DEMOS = __name__ == "__main__"


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (batch_case_share)"; several -> "exercises 2, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _signed(x: float) -> str:
    """+0.012 or -0.012, and +0.000 rather than -0.000 for a value that rounds to zero."""
    v = round(float(x), 3)
    return f"{(v if v != 0 else 0.0):+.3f}"


def _raised(kind, call):
    """Given: the exception of class `kind` that call() raised, or None if it returned normally. Another exception propagates."""
    try:
        call()
    except kind as exc:
        return exc
    return None


def _returned(call, why):
    """Given: the value call() returned. An exception your function raises becomes a failed check that says what the call was for."""
    try:
        return call()
    except NotImplementedError:
        raise
    except Exception as exc:
        raise AssertionError(f"{why}; your function raised {type(exc).__name__}: {exc}") from None


def _se(values) -> float:
    """Standard error of the mean of a list of numbers (0.0 for fewer than two)."""
    v = np.asarray(values, dtype=float)
    return float(v.std(ddof=1) / math.sqrt(len(v))) if len(v) > 1 else 0.0


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: it compares the process's resident memory with what it was when the
#    block started, and stops the block as soon as the growth passes `memory_mib`. After a memory
#    stop it looks every 5 ms and stops again at once only if the memory is still growing (a bare
#    `except:` swallowed the stop; clean-up that itself allocates over 32 MiB looks the same);
#    otherwise the stop repeats after 0.25 s, as a time stop does, so `finally:` blocks and
#    context managers can clean up. It looks after 0.01 s,
#    then every 0.02 s, and more often (down to every 2 ms) once the growth passes a quarter of
#    the limit or is fast enough to get there soon: numpy can add a gigabyte in a tenth of a
#    second. A loop that appends to a list reaches gigabytes in seconds, long before
#    any sensible time limit, so the time limit alone does not bound it. The limit is 768 MiB:
#    mybinder.org gives a whole notebook 2 GB, the heaviest lesson holds about 800 MiB before a
#    check starts, and a runaway check runs twice in "Run all" (its cell, then the board). The
#    largest growth a correct answer needs in any lesson is 512 MiB. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and (not state["fired"] or state.get("memory")):
            due.append(now + state.get("poll", 0.01))    # and look at the memory: first after 0.01 s, then as _fire says
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and (not state["fired"] or state.get("memory")) else 0
        if used_mib and not state["fired"]:              # close to the limit, or fast enough to come close soon
            rate = (grown - state.get("grown", 0)) / max(now - state.get("seen", started), 0.001)   # MiB a second
            state["grown"], state["seen"] = grown, now
            state["poll"] = (0.02 if grown <= memory_mib / 4 and rate * 0.02 <= (memory_mib - grown) / 4 else   # near: by
                                         max(0.002, min(0.01, (memory_mib - grown) / max(rate, 1.0) / 4)))  # a 1/4 of the time left
        if grown > memory_mib and state["next"] is not None and not state["fired"]:   # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
            state["memory"], state["at_stop"], state["poll"] = True, grown, 0.005   # then look every 5 ms
        elif state.get("memory") and state["fired"] and grown - state["at_stop"] > 32 and state["next"] is not None:
            state["next"], state["at_stop"] = now, grown        # still growing: a bare `except:` swallowed the stop
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 90     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 768      # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _try_rows(label: str, header: str, rows: list, footer: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a table whose rows are slow as one `_try` per row, so that the time limit covers one row and not the whole table.

    `rows` is a list of functions that each print one row and `footer` prints what follows them. If an exercise the table
    `needs` has not passed, the whole table is skipped with one line that names it, as `_try` does for any demo.
    """
    if any(_STATUS.get(name) != "passed" for name in needs):
        _try(label, footer, needs)         # prints the one line that names what the table is waiting for
        return
    print(header)
    for k, row in enumerate(rows, 1):
        _try(f"{label}, row {k}", row)
    _try(f"{label}, summary", footer)

## 1. The engine, carried over from P07-L01

The search engine of `P07-L01` is part of this lesson, and none of it is an exercise here. It is in the next cell so that
this notebook runs on its own. Its parts are a tryptic digest, peptide and b/y fragment-ion masses, a hyperscore, a search
that takes the best candidate inside a precursor-mass window, and target-decoy q-values that treat tied scores as one group.
Read it quickly: everything new in this lesson sits **around** the engine, in what goes into the database and how the
error is counted.

One fact about the chemistry matters from the start. Leucine and isoleucine have **exactly** the same mass, and glutamine
and lysine differ by only a few hundredths of a dalton. The table after the code prints both gaps, and the lesson returns
to each.

In [ ]:
class Spectrum(NamedTuple):
    """One tandem mass spectrum: `mz` is sorted ascending, `intensity` is scaled so the tallest peak is 100."""
    scan: int
    precursor_mz: float
    charge: int
    mz: np.ndarray
    intensity: np.ndarray


def neutral_mass(spectrum: Spectrum) -> float:
    """Mass of the peptide the precursor ion came from: charge times (m/z minus a proton). Given."""
    return spectrum.charge * (spectrum.precursor_mz - PROTON)


def digest(sequence: str, missed_cleavages: int = MISSED, min_len: int = MIN_LEN, max_len: int = MAX_LEN) -> list:
    """Tryptic peptides of one protein (cut after K or R unless P follows), from P07-L01. Given."""
    bounds = [0] + [i + 1 for i in range(len(sequence) - 1) if sequence[i] in "KR" and sequence[i + 1] != "P"]
    if sequence:
        bounds.append(len(sequence))
    found: dict = {}
    for a in range(len(bounds) - 1):
        for b in range(a + 1, min(a + 2 + missed_cleavages, len(bounds))):
            peptide = sequence[bounds[a]:bounds[b]]
            if min_len <= len(peptide) <= max_len:
                found.setdefault(peptide)
    return list(found)


def peptide_mass(peptide: str) -> float:
    """Neutral monoisotopic mass of a peptide: its residues plus one water. Given."""
    return float(WATER + sum(RESIDUE_MASS[r] for r in peptide))


def fragment_ions(peptide: str) -> tuple:
    """Singly charged b and y ion m/z values, b[i-1] is b_i and y[i-1] is y_i. Given."""
    masses = [RESIDUE_MASS[r] for r in peptide]
    return np.cumsum(masses[:-1]) + PROTON, np.cumsum(masses[:0:-1]) + WATER + PROTON


def match_peaks(theoretical: np.ndarray, mz: np.ndarray, intensity: np.ndarray, tol: float) -> tuple:
    """(number of theoretical ions with an observed peak within `tol`, summed intensity of those peaks). Given."""
    if len(mz) == 0 or len(theoretical) == 0:
        return 0, 0.0
    right = np.searchsorted(mz, theoretical, side="left")
    left = np.clip(right - 1, 0, len(mz) - 1)
    right = np.clip(right, 0, len(mz) - 1)
    gap_left, gap_right = np.abs(mz[left] - theoretical), np.abs(mz[right] - theoretical)
    nearest = np.where(gap_left <= gap_right, left, right)
    hit = np.minimum(gap_left, gap_right) <= tol
    return int(hit.sum()), float(intensity[nearest[hit]].sum())


def hyperscore(spectrum: Spectrum, peptide: str, tol: float) -> float:
    """ln((Ib + 1)(Iy + 1)) + ln(nb!) + ln(ny!) for one spectrum and one candidate peptide. Given."""
    b, y = fragment_ions(peptide)
    n_b, i_b = match_peaks(b, spectrum.mz, spectrum.intensity, tol)
    n_y, i_y = match_peaks(y, spectrum.mz, spectrum.intensity, tol)
    return float(math.log((i_b + 1.0) * (i_y + 1.0)) + math.lgamma(n_b + 1) + math.lgamma(n_y + 1))


class PSM(NamedTuple):
    """The best-scoring candidate for one spectrum."""
    scan: int
    peptide: str
    score: float
    is_decoy: bool
    group: str


def search_spectra(spectra: list, space: dict, precursor_ppm: float = PRECURSOR_PPM, fragment_tol: float = FRAGMENT_TOL) -> list:
    """Best candidate for each spectrum, by precursor window then hyperscore. Given.

    `space` maps a peptide to `(is_decoy, group)`. Candidates are the peptides whose mass lies within `precursor_ppm` of the
    spectrum's neutral mass; on an exact tie a decoy beats a target and then the alphabetically first peptide wins. A spectrum
    with no candidate yields no PSM.
    """
    ordered = sorted(space, key=lambda p: (peptide_mass(p), p))
    masses = np.array([peptide_mass(p) for p in ordered])
    found = []
    for spectrum in spectra:
        mass = neutral_mass(spectrum)
        half = mass * precursor_ppm / 1e6
        lo = int(np.searchsorted(masses, mass - half, side="left"))
        hi = int(np.searchsorted(masses, mass + half, side="right"))
        best = None
        for j in range(lo, hi):
            peptide = ordered[j]
            decoy = space[peptide][0]
            score = hyperscore(spectrum, peptide, fragment_tol)
            if best is None or score > best[0] or (score == best[0] and (
                    (decoy and not best[1]) or (decoy == best[1] and peptide < best[2]))):
                best = (score, decoy, peptide)
        if best is not None:
            found.append(PSM(spectrum.scan, best[2], best[0], best[1], space[best[2]][1]))
    return found


def q_values(scores, is_decoy, plus_one: bool = False) -> np.ndarray:
    """Target-decoy q-values, aligned with the input order. Given (P07-L01, exercise 7).

    For each distinct score s, from highest to lowest, count the targets T and decoys D with score >= s (a tied group is
    counted whole). The raw FDR at s is D / T (or (D + 1) / T when `plus_one`), 1.0 if T is 0, never above 1.0. The q-value
    at s is the minimum raw FDR over s and every LOWER score. Every hit, decoys included, receives the q-value of its score.
    """
    scores = np.asarray(scores, dtype=float)
    decoy = np.asarray(is_decoy, dtype=bool)
    if scores.shape != decoy.shape or scores.ndim != 1:
        raise ValueError(f"scores {scores.shape} and is_decoy {decoy.shape} must be 1-D and equal")
    n = len(scores)
    if n == 0:
        return np.zeros(0)
    order = np.argsort(-scores, kind="stable")
    s, d = scores[order], decoy[order]
    n_target, n_decoy = np.cumsum(~d), np.cumsum(d)
    raw = np.where(n_target > 0, (n_decoy + (1 if plus_one else 0)) / np.maximum(n_target, 1), 1.0)
    raw = np.minimum(raw, 1.0)
    group = np.cumsum(np.append(True, s[1:] != s[:-1])) - 1
    ends = np.append(s[1:] != s[:-1], True)
    per_group = np.minimum.accumulate(raw[ends][::-1])[::-1]
    q = np.empty(n)
    q[order] = per_group[group]
    return q


def accepted(q, is_decoy, alpha: float) -> np.ndarray:
    """Boolean mask of the TARGET hits with q-value <= alpha (the threshold itself included). Given."""
    return (np.asarray(q, dtype=float) <= alpha) & ~np.asarray(is_decoy, dtype=bool)

In [ ]:
def _isobaric_table() -> None:
    print("residue pair   mass gap (Da)    gap at 1000 Da    gap at 2000 Da")
    for a, b in (("I", "L"), ("K", "Q")):
        gap = abs(RESIDUE_MASS[a] - RESIDUE_MASS[b])
        print(f"  {a} against {b}    {gap:.6f} Da      {gap / 1000.0 * 1e6:7.2f} ppm      {gap / 2000.0 * 1e6:7.2f} ppm")
    print(f"the search window is {PRECURSOR_PPM:g} ppm of the peptide mass")


if _DEMOS:
    _isobaric_table()

## 2. A genome with called variants

Proteogenomics starts from sequence. A cohort's DNA gives a list of **variants**: a position on the genome, the base the
reference has there, and the base the sample has instead. Each world here is one chromosome carrying genes on either
strand, written as `(name, start, end, strand)` with `start` included and `end` excluded (0-based genome positions).
A gene's coding sequence (CDS) is the genome between `start` and `end`, read as written on the `+` strand and as its
**reverse complement** on the `-` strand, and it **includes the stop codon**. A variant is given on the forward strand:
`(pos, ref, alt)`, as a variant call file gives it, whichever strand the gene is on.

The simulator below is given. It writes a genome, plants a long list of candidate variants, and decides which of them
are really expressed as protein in the sample and which peptides the instrument then sees. Only a few variants are:
a called variant is a statement about DNA, and most never become a peptide anyone can see. The simulator keeps an
**answer key** for everything it decides (the key is used to grade your code and to score the tables, never by your code).

Nothing here depends on your exercises: the world comes from a seed, `SEED`, so every run prints the same numbers.

In [ ]:
_WEIGHTS = {"A": 8.0, "C": 1.5, "D": 5.5, "E": 6.5, "F": 4.0, "G": 7.0, "H": 2.3, "I": 5.5,
            "K": 5.8, "L": 9.5, "M": 2.2, "N": 4.0, "P": 4.8, "Q": 4.0, "R": 5.5, "S": 6.7,
            "T": 5.5, "V": 6.8, "W": 1.2, "Y": 3.0}   # invented, protein-like; measured from nothing
_CDF = np.cumsum([_WEIGHTS[a] for a in AMINO_ACIDS]) / sum(_WEIGHTS.values())
_SYNONYMS: dict = {}
for _codon, _aa in sorted(CODON_TABLE.items()):
    _SYNONYMS.setdefault(_aa, []).append(_codon)


class Gene(NamedTuple):
    name: str
    start: int       # first genome position of the gene (0-based)
    end: int         # one past the last position
    strand: str      # "+" or "-"


class Variant(NamedTuple):
    vid: int
    gene: str
    pos: int         # genome position (0-based, forward strand)
    ref: str         # the base the genome has there
    alt: str         # the base the sample has instead, forward strand


class VariantKey(NamedTuple):
    """The answer key for one variant: what it does and which peptides exist only because of it."""
    kind: str                # "synonymous", "missense", "nonsense", "stop-loss"
    aa_index: int            # 0-based codon number inside the CDS
    ref_aa: str
    alt_aa: str
    peptides: tuple          # ((peptide, offset of the changed residue inside it or None), ...)


class World(NamedTuple):
    genome: str
    genes: tuple
    variants: tuple
    reference_peptides: tuple    # every tryptic peptide of the reference proteome
    keys: tuple                  # one VariantKey per variant, aligned with `variants`
    spectra: tuple               # Spectrum objects, scan = position in this tuple
    source: tuple                # the peptide each spectrum was made from, None for a peptide in no database
    observable: frozenset        # ids of variants with at least one spectrum made from one of their peptides


def _norm(peptide: str) -> str:
    """Leucine and isoleucine are one residue to a mass spectrometer."""
    return peptide.replace("I", "L")


def _revcomp(seq: str) -> str:
    return "".join(COMPLEMENT[c] for c in reversed(seq))


def _cds_of(genome: str, gene: Gene) -> str:
    seq = genome[gene.start:gene.end]
    return seq if gene.strand == "+" else _revcomp(seq)


def _key_translate(cds: str) -> str:
    out = []
    for i in range(0, len(cds) - 2, 3):
        aa = CODON_TABLE[cds[i:i + 3]]
        if aa == "*":
            break
        out.append(aa)
    return "".join(out)


def _sim_spectrum(rng: np.random.Generator, peptide: str) -> Spectrum:
    charge = 2 if rng.random() < 0.6 else 3
    quality = 0.15 + 0.6 * rng.random()          # the chance that any one fragment ion shows up
    b, y = fragment_ions(peptide)
    ions = np.concatenate([b, y])
    is_y = np.concatenate([np.zeros(len(b), dtype=bool), np.ones(len(y), dtype=bool)])
    shown = rng.random(len(ions)) < quality
    signal_mz = ions[shown] + rng.normal(0.0, 0.12, int(shown.sum()))
    signal_int = np.exp(rng.normal(np.where(is_y[shown], 4.0, 3.6), 0.6))
    mass = peptide_mass(peptide) * (1.0 + rng.normal(0.0, 4e-6))   # a few ppm of measurement error
    n_noise = int(100 + 100 * rng.random())
    noise_mz = 100.0 + (min(2000.0, mass) - 100.0) * rng.random(n_noise)
    noise_int = np.exp(rng.normal(2.6, 0.8, n_noise))
    mz = np.concatenate([signal_mz, noise_mz])
    intensity = np.concatenate([signal_int, noise_int])
    order = np.argsort(mz, kind="stable")
    mz, intensity = mz[order], 100.0 * intensity[order] / intensity.max()
    return Spectrum(0, (mass + charge * PROTON) / charge, charge, mz, intensity)


def _draw_protein(rng: np.random.Generator, n: int) -> str:
    idx = np.minimum(np.searchsorted(_CDF, rng.random(n), side="right"), len(AMINO_ACIDS) - 1)
    return "M" + "".join(AMINO_ACIDS[i] for i in idx)


def make_world(world: int) -> World:
    """One simulated genome, its candidate variants, the answer key and the spectra. Given; deterministic from SEED."""
    def rng(tag: int) -> np.random.Generator:
        return np.random.default_rng([SEED, world, tag])

    r = rng(1)
    pieces, genes, cds_list, proteins, position = [], [], [], [], 0
    for g in range(N_GENES):
        spacer = "".join("ACGT"[i] for i in r.integers(0, 4, int(r.integers(30, 80))))
        pieces.append(spacer)
        position += len(spacer)
        protein = _draw_protein(r, int(r.integers(70, 160)))
        stops = _SYNONYMS["*"]
        cds = "".join(_SYNONYMS[a][int(r.integers(len(_SYNONYMS[a])))] for a in protein) + stops[int(r.integers(len(stops)))]
        strand = "+" if r.random() < 0.5 else "-"
        written = cds if strand == "+" else _revcomp(cds)
        genes.append(Gene(f"G{g:03d}", position, position + len(written), strand))
        pieces.append(written)
        position += len(written)
        cds_list.append(cds)
        proteins.append(protein)
    genome = "".join(pieces)
    reference_peptides = tuple(dict.fromkeys(p for protein in proteins for p in digest(protein)))
    reference_norm = {_norm(p) for p in reference_peptides}

    r, seen, variants, keys = rng(2), set(), [], []
    while len(variants) < N_VARIANTS:
        g = int(r.integers(N_GENES))
        t = int(r.integers(len(cds_list[g])))
        if (g, t) in seen:
            continue
        seen.add((g, t))
        gene, cds = genes[g], cds_list[g]
        pos = gene.start + t if gene.strand == "+" else gene.end - 1 - t
        ref = genome[pos]
        alt = [b for b in "ACGT" if b != ref][int(r.integers(3))]
        base = alt if gene.strand == "+" else COMPLEMENT[alt]
        mutated = cds[:t] + base + cds[t + 1:]
        ci = t // 3
        ref_aa, alt_aa = CODON_TABLE[cds[3 * ci:3 * ci + 3]], CODON_TABLE[mutated[3 * ci:3 * ci + 3]]
        kind = ("synonymous" if ref_aa == alt_aa else "stop-loss" if ref_aa == "*"
                else "nonsense" if alt_aa == "*" else "missense")
        peptides = []
        if kind in ("missense", "nonsense"):
            alt_protein = _key_translate(mutated)
            for p in digest(alt_protein):
                if _norm(p) in reference_norm:
                    continue
                offsets = [ci - s for s in range(len(alt_protein) - len(p) + 1)
                           if alt_protein.startswith(p, s) and 0 <= ci - s < len(p)]
                peptides.append((p, offsets[0] if offsets else None))
        variants.append(Variant(len(variants), gene.name, pos, ref, alt))
        keys.append(VariantKey(kind, ci, ref_aa, alt_aa, tuple(peptides)))

    r = rng(3)
    gene_on = [bool(r.random() < P_GENE) for _ in genes]
    variant_on = [bool(r.random() < P_EXPRESS) for _ in variants]
    gene_index = {gene.name: g for g, gene in enumerate(genes)}
    for v, on in zip(variants, variant_on):
        if on:
            gene_on[gene_index[v.gene]] = True

    r, spectra, source, observable = rng(4), [], [], set()

    def emit(peptide: str) -> bool:
        if r.random() >= P_DETECT:
            return False
        for _ in range(1 + int(r.poisson(1.0))):
            spectra.append(_sim_spectrum(r, peptide))
            source.append(peptide)
        return True

    for g, protein in enumerate(proteins):
        if gene_on[g]:
            for p in digest(protein):
                emit(p)
    for v, key, on in zip(variants, keys, variant_on):
        if on:
            for p, _ in key.peptides:
                if emit(p):
                    observable.add(v.vid)
    n_foreign = int(FOREIGN_SHARE * len(spectra) / (1.0 - FOREIGN_SHARE))
    for _ in range(n_foreign):
        body = _draw_protein(r, int(r.integers(7, 25)))[1:]
        peptide = body[:-1] + "KR"[int(r.integers(2))]
        spectra.append(_sim_spectrum(r, peptide))
        source.append(None)
    spectra = tuple(s._replace(scan=i) for i, s in enumerate(spectra))
    return World(genome, tuple(genes), tuple(variants), reference_peptides, tuple(keys), spectra, tuple(source),
                 frozenset(observable))


_WORLDS: dict = {}


def get_world(world: int) -> World:
    """Build world `world` once and remember it. Given."""
    if world not in _WORLDS:
        _WORLDS[world] = make_world(world)
    return _WORLDS[world]


def _gene_named(world: World, name: str) -> Gene:
    return next(g for g in world.genes if g.name == name)


def _describe_world() -> None:
    w = get_world(0)
    kinds: dict = {}
    for key in w.keys:
        kinds[key.kind] = kinds.get(key.kind, 0) + 1
    plus = sum(g.strand == "+" for g in w.genes)
    with_peptide = sum(bool(k.peptides) for k in w.keys)
    print(f"world 0: a chromosome of {len(w.genome):,} bases, {len(w.genes)} genes ({plus} on the + strand, "
          f"{len(w.genes) - plus} on the - strand)")
    print(f"  {len(w.variants):,} candidate variants: " + ", ".join(f"{n} {k}" for k, n in sorted(kinds.items())))
    print(f"  only {with_peptide} of them leave a peptide that no reference protein has, and only {len(w.observable)} "
          f"have a spectrum made from such a peptide")
    print(f"  {len(w.spectra):,} spectra, {sum(s is None for s in w.source):,} of them from peptides in no database")


if _DEMOS:
    _describe_world()

## 3. Exercise 1 — `translate()`: from a coding sequence to a protein

A ribosome reads a coding sequence three bases at a time, from the first base, and each triplet (codon) adds one amino
acid, until it meets a **stop codon**, which ends the chain and adds nothing. `CODON_TABLE` in the setup cell maps every
codon to its residue letter, with `"*"` for a stop (the standard genetic code, `claims.yaml`).

Write `translate(cds)`. It returns the protein as a string of residue letters, without the stop. A sequence with no stop
codon is translated whole. A last group of one or two bases that is not a complete codon is ignored. The empty sequence
gives the empty string. Every base of `cds` must be one of `A`, `C`, `G`, `T` (capitals): check **all** of them before you
translate, including any that come after a stop codon, and otherwise raise `ValueError`. The message must contain the
**first** character that is not a base, written as Python writes it, `repr(c)` (for a stray `N` that is `'N'`, with its
quotes), so that the message says which character and shows a space or a lower-case letter plainly.

<details><summary>💡 Hint 1 — what to think about</summary>

Which positions does each codon occupy, and where does the loop stop: at the end of the sequence, or earlier? What should a
trailing pair of bases do?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Validate the whole string first. Then step through it three at a time, look each triplet up, stop at the first `"*"`, and
join what you collected. A loop that only takes complete triplets already drops a trailing remainder.
</details>

In [ ]:
def translate(cds: str) -> str:
    """Protein encoded by a coding sequence, up to but not including the first stop codon.

    Read `cds` from its first base in codons of three, look each up in `CODON_TABLE` and stop at the first `"*"`. Ignore a
    trailing group of fewer than three bases. Raise `ValueError` if any character of `cds`, anywhere, is not one of `ACGT`;
    its message contains `repr(c)` for the first such character `c` (`'N'` for an N, with the quotes).

    Example:
        >>> translate("ATGAAATAGGGG")      # M, K, then a stop; the rest is never read
        'MK'
        >>> translate("ATGGCCA")           # M, A and a lone base
        'MA'

    Returns:
        A string of residue letters.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_translate() -> None:
    got = _returned(lambda: translate("ATGAAATAGGGG"), "a valid sequence of bases is translated, whatever its length and wherever a stop sits: no error")
    assert got == "MK", (
        f"expected 'MK', got {got!r}. The chain ends at the first stop codon (TAG here): nothing after it is read, and the "
        "stop itself is not a residue")
    got = _returned(lambda: translate("ATGGCCA"), "a valid sequence of bases is translated, whatever its length and wherever a stop sits: no error")
    assert got == "MA", f"expected 'MA', got {got!r}: a last group of fewer than three bases is ignored, not translated or an error"
    got = _returned(lambda: translate("ATGGCCAT"), "a valid sequence of bases is translated, whatever its length and wherever a stop sits: no error")
    assert got == "MA", f"expected 'MA', got {got!r}: a last group of two bases is ignored too: it is not a codon, and not an error"
    got = _returned(lambda: translate("AT"), "a valid sequence of bases is translated, whatever its length and wherever a stop sits: no error")
    assert got == "", f"expected '', got {got!r}: two bases are less than one codon"
    got = _returned(lambda: translate("ATGGCC"), "a valid sequence of bases is translated, whatever its length and wherever a stop sits: no error")
    assert got == "MA", f"expected 'MA', got {got!r}: a sequence with no stop codon is translated to its end"
    got = _returned(lambda: translate("ATGGCCTAA"), "a valid sequence of bases is translated, whatever its length and wherever a stop sits: no error")
    assert got == "MA", f"expected 'MA', got {got!r}: a stop codon that is the last codon ends the chain and adds no residue"
    assert _returned(lambda: translate(""), "the empty sequence is valid") == "", "the empty sequence gives the empty string"
    got = _returned(lambda: translate("ATTAAAGGG"), "a valid sequence of bases is translated, whatever its length and wherever a stop sits: no error")
    assert got == "IKG", f"expected 'IKG', got {got!r}: TAA at positions 2 to 4 is not in frame, the codons are ATT AAA GGG"
    got = _returned(lambda: translate("TAAATG"), "a valid sequence of bases is translated, whatever its length and wherever a stop sits: no error")
    assert got == "", f"expected '', got {got!r}: a stop in the first codon leaves an empty protein"
    got = _returned(lambda: translate("ATGTGATAG"), "a valid sequence of bases is translated, whatever its length and wherever a stop sits: no error")
    assert got == "M", f"expected 'M', got {got!r}: the first stop codon (TGA here, not only TAA or TAG) ends the chain, and a later stop changes nothing"
    got = _returned(lambda: translate("AAGCTGTTT"), "a valid sequence of bases is translated, whatever its length and wherever a stop sits: no error")
    assert got == "KLF", f"expected 'KLF', got {got!r}: read from the first base, three at a time, with CODON_TABLE"
    for bad in ("ATGN", "atg", "ATGTAAN", "AUG", "TAAxyz", "ATG ", "ATGzN"):
        first = next(c for c in bad if c not in "ACGT")
        exc = _raised(ValueError, lambda: translate(bad))
        assert exc is not None, f"translate({bad!r}) must raise ValueError: every character is checked, also after a stop codon"
        assert repr(first) in str(exc), (
            f"the message for {bad!r} must contain repr() of the first character that is not a base, {first!r} written with "
            f"its quotes, got {exc!r}")
    print("exercise 1 looks right: translate reads codons and stops at the first stop")

In [ ]:
_try("exercise 1", _check_translate)

## 4. Exercise 2 — `snv_effect()`: what a variant does to a protein

A variant is given on the forward strand of the genome, so the first job is to find which base of the **coding** sequence
it changes. On the `+` strand that is the offset `pos - start`. On the `-` strand the coding sequence is the reverse
complement, so the offset is `end - 1 - pos` and the alternative base to put into the CDS is the **complement** of `alt`.
The codon number is the offset divided by three, and comparing the reference codon with the changed one gives the effect:

- **synonymous**: the same residue, so the protein is unchanged (a stop that stays a stop counts);
- **missense**: another residue;
- **nonsense**: a residue becomes a stop, which truncates the protein;
- **stop-loss**: a stop becomes a residue, which extends the protein into sequence this lesson does not model.

Write `snv_effect(genome, gene, pos, ref, alt)`. The genome's base at `pos` must equal `ref`: if it does not (a variant
called against a different genome build, for example) return `("ref-mismatch", -1, "", "")`. Raise `ValueError` if `pos`
is outside the gene (`start <= pos < end`) or if `alt == ref`; the range check comes first, then the check of `alt`, then the
reference-base check. Return `(kind, codon_index, ref_aa, alt_aa)`, the codon index a plain Python `int`, with `"*"` for a stop.

<details><summary>💡 Hint 1 — what to think about</summary>

Draw the minus strand on paper: which end of the gene is the start of its coding sequence, and what does the base you can
read on the forward strand look like from the other side?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Do the checks in the stated order, turn the genome position into an offset in the CDS and the alternative base into the base
the CDS carries, take the three bases of the codon that holds the offset, change one of them, and look both codons up. The
four kinds come from comparing the two letters.
</details>

In [ ]:
def snv_effect(genome: str, gene: tuple, pos: int, ref: str, alt: str) -> tuple:
    """Effect of one single-nucleotide variant on the protein of the gene that holds it.

    `gene` is `(name, start, end, strand)`; its CDS is `genome[start:end]` on `"+"` and the reverse complement of it on
    `"-"`, stop codon included. `pos`, `ref` and `alt` are forward-strand. Raise `ValueError` if `pos` is not in
    `[start, end)`, then if `alt == ref`. Return `("ref-mismatch", -1, "", "")` if `genome[pos] != ref`. Otherwise the
    result is `(kind, codon_index, ref_aa, alt_aa)` with kind one of `synonymous`, `missense`, `nonsense`, `stop-loss`.

    Example:
        >>> snv_effect("CCATGAAATAGCC", ("g", 2, 11, "+"), 5, "A", "G")     # AAA (K) becomes GAA (E)
        ('missense', 1, 'K', 'E')
        >>> snv_effect("CCATGAAATAGCC", ("g", 2, 11, "+"), 5, "A", "T")     # AAA (K) becomes TAA (stop)
        ('nonsense', 1, 'K', '*')

    Returns:
        A 4-tuple.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_snv_effect() -> None:
    def inside(*args):
        """snv_effect for a position that IS in the gene: a ValueError here is the range check set one position too tight."""
        start, end = args[1][1], args[1][2]
        return _returned(lambda: snv_effect(*args), f"position {args[2]} is inside the gene [{start}, {end}): the gene holds every "
                         "position from start up to but not including end, so pos == start and pos == end - 1 are inside it")

    plus = "CCATGAAATAGCC"                      # gene (2, 11, +): ATG AAA TAG = M K stop
    gp = ("g", 2, 11, "+")
    cases = [
        ((plus, gp, 5, "A", "G"), ("missense", 1, "K", "E"), "plus strand, first base of codon 1: offset = pos - start"),
        ((plus, gp, 5, "A", "T"), ("nonsense", 1, "K", "*"), "AAA with its first base changed to T is TAA, a stop"),
        ((plus, gp, 7, "A", "G"), ("synonymous", 1, "K", "K"), "AAG codes for K just like AAA"),
        ((plus, gp, 8, "T", "C"), ("stop-loss", 2, "*", "Q"), "TAG with its first base changed to C is CAG (Q): a stop became a residue"),
        ((plus, gp, 10, "G", "A"), ("synonymous", 2, "*", "*"), "TAG to TAA is still a stop: synonymous"),
        ((plus, gp, 3, "T", "G"), ("missense", 0, "M", "R"), "ATG to AGG: codon 0, the second base"),
    ]
    for args, want, why in cases:
        got = inside(*args)
        assert got == want, f"expected {want}, got {got}: {why}"
    minus = "GG" + "CTATTTCAT" + "GG"           # gene (2, 11, -): its reverse complement is ATG AAA TAG
    gm = ("h", 2, 11, "-")
    cases = [
        ((minus, gm, 10, "T", "G"), ("missense", 0, "M", "L"), "minus strand: pos 10 is the FIRST base of the CDS, and the CDS carries the complement of alt (G gives C: CTG)"),
        ((minus, gm, 2, "C", "T"), ("synonymous", 2, "*", "*"), "minus strand, pos 2 is the LAST base of the CDS; T on the forward strand is A in the CDS: TAG to TAA"),
        ((minus, gm, 5, "T", "G"), ("missense", 1, "K", "N"), "minus strand: offset = end - 1 - pos = 5, the last base of codon 1; the CDS carries the complement of G, which is C: AAA becomes AAC (N)"),
        ((minus, gm, 7, "T", "A"), ("nonsense", 1, "K", "*"), "minus strand: the kinds are the same on either strand; offset 3, the CDS carries the complement of A, which is T: AAA becomes TAA, a stop"),
    ]
    for args, want, why in cases:
        got = inside(*args)
        assert got == want, f"expected {want}, got {got}: {why}"
    got = inside(plus, gp, 5, "C", "G")
    assert got == ("ref-mismatch", -1, "", ""), f"the genome has A at 5, not C: expected the ref-mismatch tuple, got {got}"
    assert type(inside(plus, gp, 5, "A", "G")[1]) is int, "the codon index must be a plain Python int"
    assert inside(plus, gp, 2, "A", "C") == ("missense", 0, "M", "L"), "pos == start is inside the gene"
    assert inside(minus, gm, 2, "C", "G") == ("stop-loss", 2, "*", "Y"), (
        "minus strand: pos == start is the LAST base of the CDS (TAG with its last base changed to C is TAC, Y)")
    assert inside(minus, gm, 10, "T", "C") == ("missense", 0, "M", "V"), (
        "minus strand: pos == end - 1 is the FIRST base of the CDS (ATG with its first base changed to G is GTG, V)")
    for args, why in (((plus, gp, 1, "C", "A"), "position 1 is before the gene"), ((plus, gp, 11, "C", "A"), "end is excluded"),
                      ((plus, gp, 5, "A", "A"), "alt equals ref")):
        assert _raised(ValueError, lambda: snv_effect(*args)) is not None, f"ValueError expected: {why}"
    for args, why in (((plus, gp, 1, "G", "G"), "alt == ref outside the gene"), ((plus, gp, 1, "G", "C"), "a wrong reference base outside the gene"),
                      ((plus, gp, 5, "C", "C"), "alt == ref with a wrong reference base too: the alt check comes before the reference-base check")):
        assert _raised(ValueError, lambda: snv_effect(*args)) is not None, (
            f"ValueError expected: {why} (the range check comes first, then alt against ref, then the reference base)")
    print("exercise 2 looks right: strands, codon positions and the four kinds are handled")

In [ ]:
_try("exercise 2", _check_snv_effect)

## 5. Exercise 3 — `novel_peptides()`: the peptides that exist only because of the variant

A search can only find a peptide that is in its database. To find evidence for a variant you add the **variant
peptides**: the tryptic peptides of the changed protein that no reference protein produces. A peptide that spans the changed
residue is novel, and so is one that exists only because the change **created or removed a cleavage site** (a `K` or `R`
appearing or disappearing moves a cut, and the neighbouring peptides change with it). Peptides far from the change are the
reference's own and are not evidence for anything.

"Novel" has a second subtlety. Leucine and isoleucine weigh exactly the same (the table in section 1 prints the gap), so a
search that works on masses cannot tell them apart, and a variant that swaps one for the other produces a peptide that is
**indistinguishable** from the reference's to this engine. It is not novel in any sense this search can use. Write
`novel_peptides(reference_peptides, alt_protein)`: digest `alt_protein` with `digest` (the given function) and return, in
digest order, the peptides for which **no reference peptide** is the same once `I` is read as `L` (compare on that basis,
return the peptide as it is). `reference_peptides` is any iterable of peptide strings (it may be a generator, which can be
read only once) and is the digest of the **whole**
reference proteome, not of one gene: a variant peptide that another protein already produces is not specific to the variant.
Speed matters a little here: the table below this exercise calls your function for hundreds of variants per world against a
reference of thousands of peptides, and every cell is stopped at its time limit, so look the peptides up in a set (or a dict)
built once per call. Scanning the whole reference list for every peptide is correct, but far slower.

<details><summary>💡 Hint 1 — what to think about</summary>

What do you compare each peptide of the changed protein against, and in what form? What happens to a variant peptide that
differs from a reference peptide only by I and L? And what if `reference_peptides` is a generator?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Turn the reference peptides into a set of their I-as-L forms once, digest the changed protein, and keep each peptide whose
I-as-L form is not in the set.
</details>

In [ ]:
def novel_peptides(reference_peptides, alt_protein: str) -> list:
    """Tryptic peptides of `alt_protein` that no reference peptide matches, reading I and L as the same residue.

    Digest `alt_protein` with `digest` (the defaults) and keep, in the order `digest` gives them, each peptide whose form with
    every `I` replaced by `L` differs from that form of every reference peptide. `reference_peptides` is any iterable of
    strings (read it once; it may be a generator). Return the peptides unchanged, as a list.

    Example:
        >>> novel_peptides(["AAAAAAK"], "AAAAAAKCCCCCCCR")      # the peptide after the new cut is novel
        ['AAAAAAKCCCCCCCR', 'CCCCCCCR']
        >>> novel_peptides(["IIIIIIK"], "LLLLLLK")               # indistinguishable from the reference peptide
        []

    Returns:
        A list of peptide strings.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_novel_peptides() -> None:
    got = novel_peptides(["AAAAAAK"], "AAAAAAKCCCCCCCR")
    assert got == ["AAAAAAKCCCCCCCR", "CCCCCCCR"], (
        f"expected ['AAAAAAKCCCCCCCR', 'CCCCCCCR'], got {got!r}: keep every peptide of the changed protein that the "
        "reference does not produce, in digest order")
    got = novel_peptides(["IIIIIIK"], "LLLLLLK")
    assert got == [], (
        f"expected [], got {got!r}: I and L weigh exactly the same, so a peptide that differs from a reference peptide only "
        "by I against L cannot be told apart from it")
    got = novel_peptides(["LLLLLLK"], "IIIIIIK")
    assert got == [], f"expected [], got {got!r}: the I-as-L reading works in both directions"
    got = novel_peptides(["AAAAAAK"], "IAAAAAKGGGGGGR")
    assert got == ["IAAAAAK", "IAAAAAKGGGGGGR", "GGGGGGR"], (
        f"expected ['IAAAAAK', 'IAAAAAKGGGGGGR', 'GGGGGGR'], got {got!r}: return each novel peptide as it is, with its I, not in its "
        "I-as-L form")
    got = novel_peptides(["AAAAAAK"], "AAAAAAR")
    assert got == ["AAAAAAR"], f"expected ['AAAAAAR'], got {got!r}: only I and L are read as one residue; K and R are different residues"
    got = novel_peptides(iter(["GGGGGGR", "AAAAAAK"]), "AAAAAAKGGGGGGR")
    assert got == ["AAAAAAKGGGGGGR"], (
        f"expected ['AAAAAAKGGGGGGR'], got {got!r}: reference_peptides may be a generator, which can be read only once")
    got = novel_peptides(("AAAAAAK", "CCCCCCR"), "AAAAAAKCCCCCCR")
    assert got == ["AAAAAAKCCCCCCR"] and type(got) is list, (
        f"expected the list ['AAAAAAKCCCCCCR'], got {got!r}: peptides found anywhere in the reference proteome are not specific "
        "to the variant")
    got = novel_peptides([], "AAAAAAKCCCCCCCR")
    assert got == ["AAAAAAK", "AAAAAAKCCCCCCCR", "CCCCCCCR"], f"expected all three peptides, got {got!r}: an empty reference makes every peptide novel"
    got = novel_peptides([], "LAAAAAKIAAAAAK")
    assert got == ["LAAAAAK", "LAAAAAKIAAAAAK", "IAAAAAK"], (
        f"expected all three peptides, got {got!r}: a peptide of the changed protein is compared with the reference peptides only, "
        "never with the other peptides of its own digest, so two that differ only by I and L are both kept")
    got = novel_peptides(["AAAQPAAAK"], "AAAKPAAAK")
    assert got == ["AAAKPAAAK"], (
        f"expected ['AAAKPAAAK'], got {got!r}: only I and L are read as one residue. K and Q are near-isobaric but different residues, "
        "so a peptide with a K against a reference Q is novel (K before P makes no cut, so this protein is one peptide)")
    heavy_ref, heavy_alt = "W" * 10 + "QP" + "W" * 10 + "K", "W" * 10 + "KP" + "W" * 10 + "K"
    got = novel_peptides([heavy_ref], heavy_alt)
    assert got == [heavy_alt], (
        f"expected [{heavy_alt!r}], got {got!r}: only I and L are read as one residue, however close K and Q are in mass. This pair "
        "is about 9 ppm apart, inside the search window of the search below, and it is still two different peptides")
    got = novel_peptides([], "GGGGGKAAAAAAAR")
    assert got == ["GGGGGKAAAAAAAR", "AAAAAAAR"], (
        f"expected ['GGGGGKAAAAAAAR', 'AAAAAAAR'], got {got!r}: use digest with its defaults. The peptide GGGGGK is one residue "
        "shorter than the shortest peptide digest keeps (see digest.__defaults__)")
    got = novel_peptides([], "A" * 29 + "K")
    assert got == ["A" * 29 + "K"], (
        f"expected the one peptide AAAA...K, got {got!r}: use digest with its defaults, which keep a peptide of this length")
    got = novel_peptides([], "A" * 30 + "K")
    assert got == [], (
        f"expected [], got {got!r}: this peptide is one residue longer than the longest peptide digest keeps with its defaults "
        "(see digest.__defaults__)")
    got = novel_peptides([], "AAAAAAKCCCCCCKGGGGGGR")
    assert got == ["AAAAAAK", "AAAAAAKCCCCCCK", "CCCCCCK", "CCCCCCKGGGGGGR", "GGGGGGR"], (
        f"expected five peptides, got {got!r}: use digest with its defaults, which allow one missed cleavage, so a peptide made of "
        "three pieces is not kept (see digest.__defaults__)")
    wrong = []
    for a in "ACDEFGHIKLMNPQRSTVWY":
        for b in "ACDEFGHIKLMNPQRSTVWY":
            if a != b:
                ref, alt = "GGG" + a + "PGGGGK", "GGG" + b + "PGGGGK"
                if novel_peptides([ref], alt) != ([] if {a, b} == {"I", "L"} else [alt]):
                    wrong.append((a, b))
    assert not wrong, (
        f"these pairs of residues were read wrongly (reference residue, variant residue): {wrong[:6]}. Only I and L are read as one "
        "residue; every other pair of different residues makes the peptide novel, and I against L does not")
    assert novel_peptides([], "") == [], "an empty protein has no peptides"
    print("exercise 3 looks right: only peptides no reference peptide can explain are kept")

In [ ]:
_try("exercise 3", _check_novel_peptides)

In [ ]:
def _alt_protein(world: World, v: Variant) -> str:
    """The protein of the variant allele, using YOUR `translate`. Given: the gene's CDS with the one base changed."""
    gene = _gene_named(world, v.gene)
    cds = _cds_of(world.genome, gene)
    t = v.pos - gene.start if gene.strand == "+" else gene.end - 1 - v.pos
    base = v.alt if gene.strand == "+" else COMPLEMENT[v.alt]
    return translate(cds[:t] + base + cds[t + 1:])


_CHECK_EVERY = 16     # of the variants whose peptide list is routine, the demo calls YOUR novel_peptides for one in this many


def _variant_row(world_id: int) -> None:
    """One row: your exercises 1 to 3 against the answer key for every variant of one world. Your effect is compared for all
    of them; your peptide list for every variant where it is not routine (nonsense, or the key finds no novel peptide, as for a
    swap of I and L) and for every `_CHECK_EVERY`-th of the others, because the list is the slow part: it is a call of your
    function for each variant, against the whole reference proteome."""
    w = get_world(world_id)
    known = set(w.reference_peptides)                    # built once per world, not once per variant
    kind_ok = pep_ok = checked = naive_extra = 0
    first_effect = first_peptides = None                 # the first variant where your answer and the key disagree
    for number, (v, key) in enumerate(zip(w.variants, w.keys)):
        gene = _gene_named(w, v.gene)
        effect = snv_effect(w.genome, tuple(gene), v.pos, v.ref, v.alt)
        same = effect == (key.kind, key.aa_index, key.ref_aa, key.alt_aa)
        kind_ok += same
        if not same and first_effect is None:
            first_effect = (number, v, (key.kind, key.aa_index, key.ref_aa, key.alt_aa), effect)
        if key.kind not in ("missense", "nonsense"):
            continue
        if not (key.kind == "nonsense" or not key.peptides or number % _CHECK_EVERY == 0):
            continue
        alt = _alt_protein(w, v)                         # your translate, once for this variant
        mine = novel_peptides(w.reference_peptides, alt)
        checked += 1
        want = [p for p, _ in key.peptides]
        pep_ok += mine == want
        if mine != want and first_peptides is None:
            first_peptides = (number, v, want, mine)
        naive_extra += len([p for p in digest(alt) if p not in known]) - len(mine)
    n = len(w.variants)
    print(f"world {world_id}: {n:,} variants, effect agrees with the key for {kind_ok:,}, peptide list for {pep_ok:,} of the "
          f"{checked:,} checked; ignoring the I/L rule would add {naive_extra} peptides")
    # The board reads this: a table that disagrees with the key turns the exercise it tests red, so a rule that only the table
    # checks can never leave the board green.
    row = f"section 5, world {world_id}"
    for exercise, found in (("exercise 2", first_effect), ("exercise 3", first_peptides)):
        _ROW_FAILS.setdefault(exercise, {}).pop(row, None)
        if found is not None:
            _ROW_FAILS[exercise][row] = "disagrees with the answer key"
    if first_effect is not None:
        number, v, want_effect, got_effect = first_effect
        raise AssertionError(
            f"variant {number} of world {world_id} ({v.gene}, position {v.pos}, {v.ref} to {v.alt}): the key says {want_effect}, "
            f"your snv_effect says {got_effect}; look at the strand, the codon position and the ref-mismatch rule, and at translate")
    if first_peptides is not None:
        number, v, want_list, got_list = first_peptides
        raise AssertionError(
            f"variant {number} of world {world_id} ({v.gene}, position {v.pos}, {v.ref} to {v.alt}): the key lists {want_list}, "
            f"your novel_peptides lists {got_list}; only I and L are read as one residue (K and Q are not), and the peptides are "
            "those of digest with its defaults (look at digest.__defaults__ for the lengths it keeps)")


def _variant_rows_footer() -> None:
    w = get_world(0)
    empty: dict = {}
    for key in w.keys:
        if not key.peptides:
            empty[key.kind] = empty.get(key.kind, 0) + 1
    swaps = sum(1 for key in w.keys if key.kind == "missense" and not key.peptides and {key.ref_aa, key.alt_aa} == {"I", "L"})
    none = sum(empty.values())
    print(f"in world 0, {none} of {len(w.variants):,} variants ({100.0 * none / len(w.variants):.1f}%) leave no novel peptide, "
          f"so no spectrum can ever support them: " + ", ".join(f"{n} {k}" for k, n in sorted(empty.items()))
          + f"; {swaps} of the missense ones swap I and L")

if _DEMOS:
    _try_rows("section 5", "your functions against the answer key, world by world:",
              [lambda w=w: _variant_row(w) for w in range(N_WORLDS)], _variant_rows_footer,
              needs=("exercise 1", "exercise 2", "exercise 3"))

## 6. Exercise 4 — `build_space()`: a database whose decoys know where they came from

The search needs one database that holds the reference peptides, the variant peptides and **decoys**. A decoy is a peptide
that cannot be in the sample, made from a real one, so that wrong matches to it can be counted: the engine treats a
decoy hit as a stand-in for a wrong target hit (P07-L01). Here the decoy of a peptide is its sequence **reversed except for
the last residue**, so it keeps the final `K` or `R` and exactly the same mass.

Write `build_space(reference_peptides, variant_peptides)`. It returns a dict that maps each peptide string to
`(is_decoy, group)`, with `group` equal to `"ref"` or `"var"`:

- each reference peptide is a target in group `"ref"`;
- each variant peptide is a target in group `"var"`, unless it is also a reference peptide (exactly the same string), in
  which case it stays `"ref"` and appears once;
- each target has a decoy, which is **in the same group as the target it came from**: that is what lets the variant
  peptides' wrong matches be counted on their own, in exercise 5;
- a decoy that equals any target is dropped (the target wins), and no peptide is ever both target and decoy.

Both arguments may be any iterable of strings (each is read once), a peptide listed twice is one target, and the arguments
are left unchanged. Strings are compared exactly here: `I` and `L` are not merged when the database is built.

<details><summary>💡 Hint 1 — what to think about</summary>

What are the two kinds of target, and which wins when a string is in both lists? From which target does each decoy come,
and what must it inherit? What should happen when a decoy happens to be a real target, as it is for a palindromic peptide?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Fill in the targets first (reference, then variant peptides not yet there), so that you can test every decoy against all of
them. Then make one decoy per target, give it its target's group, and add it only if its string is not already a target.
</details>

In [ ]:
def build_space(reference_peptides: list, variant_peptides: list) -> dict:
    """Search database: target and decoy peptides, each with a group.

    Returns a dict `peptide -> (is_decoy, group)`. Reference peptides are targets of group `"ref"`; variant peptides not
    already present are targets of group `"var"`. The decoy of a peptide `p` is `p[:-1][::-1] + p[-1]` and carries the group
    of `p`. A decoy whose string equals any target is not added. `is_decoy` is a plain `bool`.

    Example:
        >>> sorted(build_space(["AGCDEFK"], ["WYHMNPR"]).items())
        [('AGCDEFK', (False, 'ref')), ('FEDCGAK', (True, 'ref')), ('PNMHYWR', (True, 'var')), ('WYHMNPR', (False, 'var'))]

    Returns:
        A dict with one entry per distinct target and one per surviving decoy.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_build_space() -> None:
    got = build_space(["AGCDEFK"], ["WYHMNPR"])
    want = {"AGCDEFK": (False, "ref"), "FEDCGAK": (True, "ref"), "WYHMNPR": (False, "var"), "PNMHYWR": (True, "var")}
    assert got == want, (
        f"expected {want}, got {got}. Each decoy is the peptide reversed except its last residue, and it belongs to the group "
        "of the peptide it came from (a decoy of a variant peptide is a variant decoy)")
    got = build_space(["AGCDEFK", "WYHMNPR"], ["WYHMNPR", "DDDDDDK"])
    want = {"AGCDEFK": (False, "ref"), "FEDCGAK": (True, "ref"), "WYHMNPR": (False, "ref"), "PNMHYWR": (True, "ref"), "DDDDDDK": (False, "var")}
    assert got == want, (
        f"expected {want}, got {got}. A variant peptide that is also a reference peptide is a reference target, so its decoy is a "
        "reference decoy (the group of the target it came from, not of the list it was found in); DDDDDDK is new, so it is a variant "
        "target, and as a palindrome of its body it has no decoy")
    got = build_space(["AGCDEFK", "FEDCGAK"], [])
    assert got == {"AGCDEFK": (False, "ref"), "FEDCGAK": (False, "ref")}, (
        f"expected the two targets and no decoys, got {got}: each decoy is the other target's sequence, and a decoy that equals "
        "a target is dropped, because the target wins")
    got = build_space(["AAAAAAK"], ["GGGGGGR"])
    assert got == {"AAAAAAK": (False, "ref"), "GGGGGGR": (False, "var")}, (
        f"expected only the two targets, got {got}: reversing all but the last residue of a peptide whose body is a palindrome "
        "gives the peptide itself, which is a target, so there is no decoy")
    got = build_space(["AGCDEFK"], ["FEDCGAK"])
    assert got == {"AGCDEFK": (False, "ref"), "FEDCGAK": (False, "var")}, (
        f"expected two targets and no decoys, got {got}: the decoy of the reference peptide is a variant target and the other way "
        "round")
    got = build_space(["AGCIDFK"], ["FDLCGAK"])
    assert got == {"AGCIDFK": (False, "ref"), "FDLCGAK": (False, "var"), "FDICGAK": (True, "ref"), "AGCLDFK": (True, "var")}, (
        f"got {got}: a decoy is dropped only when it is exactly the string of a target; I and L are not merged when the database is built")
    assert all(type(v[0]) is bool for v in build_space(["AGCDEFK"], ["WYHMNPR"]).values()), "is_decoy must be a plain bool"
    assert build_space([], []) == {}, "no peptides, no database"
    got = build_space(["AGCDEFK"], [])
    assert got == {"AGCDEFK": (False, "ref"), "FEDCGAK": (True, "ref")}, f"got {got}: with no variant peptides the reference targets still have their decoys"
    got = build_space([], ["WYHMNPR"])
    assert got == {"WYHMNPR": (False, "var"), "PNMHYWR": (True, "var")}, f"got {got}: with no reference peptides the variant targets are still there, with their decoys"
    ref, var = ["AGCDEFK", "AGCDEFK"], ["WYHMNPR", "WYHMNPR"]
    got = build_space(iter(ref), iter(var))
    assert got == build_space(["AGCDEFK"], ["WYHMNPR"]), (
        f"got {got}: the peptides may come from any iterable, and one listed twice is one target")
    keep = (list(ref), list(var))
    build_space(ref, var)
    assert (ref, var) == keep, "do not change the lists you are given"
    print("exercise 4 looks right: targets by group, decoys that keep their target's group")

In [ ]:
_try("exercise 4", _check_build_space)

In [ ]:
def _variant_peptides(world: World) -> list:
    """Every variant peptide in the answer key, once, in variant order. The lesson's own list, so that exercises 4 to 7 do not
    depend on exercise 3 (the table in section 5 told you whether yours agrees)."""
    return list(dict.fromkeys(p for key in world.keys for p, _ in key.peptides))


_SEARCH: dict = {}


def searched(world_id: int, with_variants: bool) -> tuple:
    """`(space, psms)` for one world, from YOUR `build_space`. Given; the search runs once and is remembered (the key includes
    your function, so editing exercise 4 and running it again searches again)."""
    key = (build_space, world_id, with_variants)
    if key not in _SEARCH:
        w = get_world(world_id)
        variants = _variant_peptides(w) if with_variants else []
        space = build_space(list(w.reference_peptides), variants)
        _SEARCH[key] = (space, search_spectra(list(w.spectra), space))
    return _SEARCH[key]


def _wrong(w: World, psm: PSM) -> bool:
    """True if the peptide a spectrum was matched to is not the one it was made from (I and L are one residue)."""
    made_from = w.source[psm.scan]
    return made_from is None or _norm(made_from) != _norm(psm.peptide)


def _pooled(psms: list) -> np.ndarray:
    """Mask of accepted targets at ALPHA when the error rate is estimated over all PSMs together. Given."""
    scores = [p.score for p in psms]
    decoy = [p.is_decoy for p in psms]
    return accepted(q_values(scores, decoy), decoy, ALPHA)


def _search_row(world_id: int) -> None:
    w = get_world(world_id)
    _, only = searched(world_id, False)
    _, aug = searched(world_id, True)
    got_only, got_aug = _pooled(only), _pooled(aug)
    wrong_only = sum(_wrong(w, p) for p, a in zip(only, got_only) if a)
    wrong_aug = sum(_wrong(w, p) for p, a in zip(aug, got_aug) if a)
    var_acc = [p for p, a in zip(aug, got_aug) if a and p.group == "var"]
    var_wrong = sum(_wrong(w, p) for p in var_acc)
    from_variants = {p for p in _variant_peptides(w)}
    gained = sum(1 for p in var_acc if not _wrong(w, p))
    in_data = sum(1 for s in w.source if s in from_variants)
    print(f"world {world_id}: reference only {int(got_only.sum()):>5} accepted, {wrong_only:>3} wrong | "
          f"with variants {int(got_aug.sum()):>5} accepted, {wrong_aug:>3} wrong | variant PSMs {len(var_acc):>4}, "
          f"{var_wrong:>3} wrong, {gained:>3} right of {in_data:>3} variant spectra")


def _search_footer() -> None:
    tot_acc = tot_wrong = tot_all = 0
    for world_id in range(N_WORLDS):
        w = get_world(world_id)
        _, aug = searched(world_id, True)
        mask = _pooled(aug)
        tot_all += int(mask.sum())
        for p, a in zip(aug, mask):
            if a and p.group == "var":
                tot_acc += 1
                tot_wrong += _wrong(w, p)
    print(f"over {N_WORLDS} worlds, {tot_acc} of {tot_all:,} accepted PSMs ({100.0 * tot_acc / tot_all:.1f}%) are variant PSMs, "
          f"and {tot_wrong} of those ({100.0 * tot_wrong / max(tot_acc, 1):.1f}%) are wrong; the pooled error rate was set to "
          f"{100.0 * ALPHA:g}%")


if _DEMOS:
    _try_rows("section 6", "pooled error control at one percent, with and without the variant peptides in the database:",
              [lambda w=w: _search_row(w) for w in range(N_WORLDS)], _search_footer, needs=("exercise 4",))

## 7. Exercise 5 — `group_q_values()`: an error rate for the variant peptides alone

The pooled estimate counts decoys over **all** hits and says that about one percent of the accepted PSMs are wrong. That is an
average over a very uneven crowd: almost all the accepted hits are reference peptides that are really in the sample, while
the variant database is mostly peptides that are not. Wrong hits come in proportion to the size of the database they can land
in, not to the number of true hits, so the variant group has far more wrong hits **per accepted hit** than the pooled rate
shows. The remedy (`claims.yaml`) is a separate estimate for the group, made from the group's own decoys.

Write `group_q_values(scores, is_decoy, group, min_decoys=MIN_DECOYS, plus_one=False)`. For each distinct label in `group`,
take only the PSMs with that label and compute their q-values with the given `q_values` (`plus_one` passed on). Return one
value per PSM, **in the order of the input**. A group with **fewer than `min_decoys`** decoy PSMs cannot be estimated: every
PSM of that group, decoys included, gets the q-value `1.0`. A group with exactly `min_decoys` decoys is estimated, and so is every
group when `min_decoys` is `0`; a group with no decoy at all is "fewer" whenever `min_decoys` is `1` or more. Every estimated
group, one with no decoy included, gets exactly what `q_values` returns for it with `plus_one` passed on (with `plus_one` a group
of targets only does not get `0.0`). Raise
`ValueError` unless the three inputs are one-dimensional and have the same length (no PSMs at all is fine and gives an empty
array). The PSMs come in any order, and the scores may be integers. Return a float array, and leave the inputs unchanged. The
group labels can be any strings.

<details><summary>💡 Hint 1 — what to think about</summary>

What do you compute for each group, and how do the results get back to the right positions of the output? What does a group
with almost no decoys tell you about its error rate?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Make the three inputs arrays and check their shapes. Start from an array of ones. For each distinct label, select its
positions with a boolean mask, count its decoys, and if there are enough, write the q-values of those positions back into
the same positions.
</details>

In [ ]:
def group_q_values(scores, is_decoy, group, min_decoys: int = MIN_DECOYS, plus_one: bool = False) -> np.ndarray:
    """Target-decoy q-values computed inside each group separately.

    For each distinct label in `group`, the PSMs with that label get the `q_values` of those PSMs alone (`plus_one` is
    passed on). If the group has fewer than `min_decoys` decoy PSMs (none at all counts, unless `min_decoys` is 0), all its
    PSMs get `1.0`. The result is aligned with the input, whatever the order of the PSMs. Raise `ValueError` unless the three
    inputs are 1-D and equally long (empty is fine).

    Example:
        >>> group_q_values([9, 8, 7, 6, 5, 4], [False, False, False, True, False, True],
        ...                ["a", "b", "a", "a", "b", "b"], min_decoys=1).tolist()
        [0.0, 0.0, 0.0, 0.5, 0.0, 0.5]

    Returns:
        A float array with one q-value per PSM.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_group_q_values() -> None:
    scores, decoy, grp = [9, 8, 7, 6, 5, 4], [False, False, False, True, False, True], ["a", "b", "a", "a", "b", "b"]
    got = group_q_values(scores, decoy, grp, min_decoys=1)
    assert isinstance(got, np.ndarray) and got.dtype.kind == "f", (
        f"expected an array of floats, got {type(got).__name__} {getattr(got, 'dtype', '')}: return a float numpy array, one q-value per PSM")
    assert got.tolist() == [0.0, 0.0, 0.0, 0.5, 0.0, 0.5], (
        f"expected [0.0, 0.0, 0.0, 0.5, 0.0, 0.5], got {got.tolist()}. Each group is ranked and counted on its own (a decoy in "
        "group a does not count against group b), and the answer is written back in the order of the input")
    got = group_q_values(scores, decoy, ["a"] * 6, min_decoys=1)
    assert np.allclose(got, q_values(scores, decoy), rtol=0, atol=1e-12), (
        "with one group the result must equal the pooled q_values")
    got = group_q_values(scores, decoy, grp, min_decoys=2)
    assert got.tolist() == [1.0] * 6, (
        f"expected all 1.0, got {got.tolist()}: each group holds one decoy, fewer than min_decoys = 2, so its error rate "
        "cannot be estimated and every PSM of it gets 1.0")
    got = group_q_values([5.0, 4.0, 3.0], [False, False, True], ["x", "x", "x"], min_decoys=1)
    assert got.tolist() == [0.0, 0.0, 0.5], f"expected [0.0, 0.0, 0.5], got {got.tolist()}"
    got = group_q_values([4, 9, 6, 5, 8, 7], [True, False, True, False, False, False], ["b", "a", "a", "b", "b", "a"], min_decoys=1)
    assert got.tolist() == [0.5, 0.0, 0.5, 0.0, 0.0, 0.0], (
        f"expected [0.5, 0.0, 0.5, 0.0, 0.0, 0.0], got {got.tolist()}: the same PSMs as the first example in another order; the PSMs "
        "are not sorted by score, and integer scores are fine")
    got = group_q_values([5.0, 3.0, 3.0, 3.0], [False, False, True, False], ["x"] * 4, min_decoys=1)
    assert np.allclose(got, [0.0, 1 / 3, 1 / 3, 1 / 3], rtol=0, atol=1e-12), (
        f"expected [0.0, 1/3, 1/3, 1/3], got {got.tolist()}: three PSMs tie at 3.0 and are counted together, as q_values counts them")
    got = group_q_values([5.0, 4.0], [False, False], ["x", "x"], min_decoys=1)
    assert got.tolist() == [1.0, 1.0], (
        f"expected [1.0, 1.0], got {got.tolist()}: a group of targets only has no decoys, fewer than min_decoys = 1, so it "
        "cannot be estimated (an error rate of 0 from no decoys is not an estimate)")
    got = group_q_values([5.0, 4.0, 3.0, 2.0, 9.0, 8.0], [False, True, True, False, False, False], ["g", "g", "g", "g", "h", "h"], min_decoys=2)
    assert got.tolist() == [0.0, 1.0, 1.0, 1.0, 1.0, 1.0], (
        f"expected [0.0, 1.0, 1.0, 1.0, 1.0, 1.0], got {got.tolist()}: group g has two decoys, exactly min_decoys, so it is "
        "estimated; group h has none, so both its PSMs get 1.0")
    got = group_q_values([5.0, 4.0], [False, False], ["x", "x"], min_decoys=0)
    assert got.tolist() == [0.0, 0.0], (
        f"expected [0.0, 0.0], got {got.tolist()}: with min_decoys = 0 a group of targets only is estimated, and has no decoys")
    got = group_q_values([5.0, 4.0, 3.0], [False, False, True], ["x", "x", "x"], min_decoys=1, plus_one=True)
    assert np.allclose(got, [0.5, 0.5, 1.0], rtol=0, atol=1e-12), (
        f"expected [0.5, 0.5, 1.0] with plus_one, got {got.tolist()}: pass plus_one on to q_values")
    got = group_q_values([5.0, 4.0], [False, False], ["x", "x"], min_decoys=0, plus_one=True)
    assert isinstance(got, np.ndarray) and got.dtype.kind == "f" and np.allclose(got, q_values([5.0, 4.0], [False, False], True), rtol=0, atol=1e-12), (
        f"expected {q_values([5.0, 4.0], [False, False], True).tolist()}, got {got.tolist()} (a {getattr(got, 'dtype', type(got))} result): an estimated "
        "group always goes through q_values with plus_one passed on, also when it has no decoy (plus_one adds one to the decoy count, "
        "so a group of targets only does not get 0.0), and the result is an array of floats")
    got = group_q_values([5.0, 4.0, 3.0, 2.0, 9.0, 8.0, 7.0], [False, True, False, False, False, False, False],
                         ["g", "g", "g", "g", "h", "h", "h"], min_decoys=0, plus_one=True)
    want = np.concatenate([q_values([5.0, 4.0, 3.0, 2.0], [False, True, False, False], True), q_values([9.0, 8.0, 7.0], [False] * 3, True)])
    assert np.allclose(got, want, rtol=0, atol=1e-12), (
        f"expected {want.tolist()}, got {got.tolist()}: group g has one decoy and group h has none; both are estimated with plus_one, "
        "and plus_one counts one extra decoy in each")
    shapes = [([5.0, 4.0], [False, False], ["x", "x"]),                                         # targets only
              ([3.0, 2.0], [True, True], ["d", "d"]),                                            # decoys only
              ([9, 8, 7, 6, 5, 4, 3], [False, False, False, True, False, True, False], ["a", "b", "a", "a", "b", "b", "a"]),
              ([6, 6, 5, 5, 4, 3, 3, 2, 9, 8, 7, 1], [False, True, False, False, True, False, True, False, False, False, False, True],
               ["p", "p", "p", "q", "q", "q", "q", "q", "r", "r", "r", "p"])]                   # three groups with 0, 2 and 3 decoys, ties
    for m in range(5):
        for plus in (False, True):
            for sc, dc, gp in shapes:
                got = group_q_values(sc, dc, gp, m, plus)
                want = np.ones(len(sc))
                for label in sorted(set(gp)):
                    at = [i for i in range(len(sc)) if gp[i] == label]
                    if sum(dc[i] for i in at) >= m:
                        want[at] = q_values([sc[i] for i in at], [dc[i] for i in at], plus)
                assert isinstance(got, np.ndarray) and got.dtype.kind == "f" and np.allclose(got, want, rtol=0, atol=1e-12), (
                    f"min_decoys {m}, plus_one {plus}, groups {gp}: expected {want.tolist()}, got {np.asarray(got).tolist()}: a group with "
                    "at least min_decoys decoys (none at all counts when min_decoys is 0) gets exactly its own q_values with plus_one "
                    "passed on, any other group gets 1.0, and the result is an array of floats")
    sc, dc, gp = np.array([3.0, 1.0, 2.0]), np.array([False, True, False]), np.array(["b", "b", "a"])
    keep = (sc.copy(), dc.copy(), gp.copy())
    group_q_values(sc, dc, gp, min_decoys=1)
    assert all(np.array_equal(a, b) for a, b in zip((sc, dc, gp), keep)), "do not change the arrays you are given"
    assert group_q_values([], [], [], min_decoys=1).tolist() == [], "no PSMs, no q-values"
    got = group_q_values([5.0, 4.0, 3.0], [False, True, False], ["y"] * 3)
    assert got.tolist() == [1.0, 1.0, 1.0], (
        f"got {got.tolist()}: when min_decoys is not given it is MIN_DECOYS (10), and a group of three PSMs cannot reach it")
    for k in (MIN_DECOYS - 1, MIN_DECOYS):
        got = group_q_values([float(60 - i) for i in range(40 + k)], [False] * 40 + [True] * k, ["y"] * (40 + k))
        assert (got.min() == 1.0) == (k < MIN_DECOYS), (
            f"with {k} decoys and min_decoys not given, the lowest q-value is {got.min()}: the default is MIN_DECOYS ({MIN_DECOYS}), so a "
            f"group with {MIN_DECOYS - 1} decoys gets 1.0 and one with {MIN_DECOYS} is estimated")
    for bad in (([1.0, 2.0], [False], ["a", "a"]), ([1.0, 2.0], [False, True], ["a"]), ([[1.0, 2.0]], [[False, True]], [["a", "a"]])):
        assert _raised(ValueError, lambda: group_q_values(*bad, min_decoys=1)) is not None, (
            f"group_q_values{bad} must raise ValueError: the three inputs must be 1-D and equally long")
    print("exercise 5 looks right: q-values per group, aligned with the input, and no estimate where there are too few decoys")

In [ ]:
_try("exercise 5", _check_group_q_values)

In [ ]:
def _subset_mask(psms: list) -> np.ndarray:
    """Accepted targets at ALPHA when each group has its own error estimate (YOUR `group_q_values`). Given."""
    decoy = [p.is_decoy for p in psms]
    q = group_q_values([p.score for p in psms], decoy, [p.group for p in psms], MIN_DECOYS)
    return accepted(q, decoy, ALPHA)


def _fdp(w: World, psms: list, mask) -> tuple:
    """(number accepted, number of them wrong) among the PSMs selected by `mask`."""
    chosen = [p for p, m in zip(psms, mask) if m]
    return len(chosen), sum(_wrong(w, p) for p in chosen)


def _compare_row(world_id: int) -> None:
    w = get_world(world_id)
    _, psms = searched(world_id, True)
    is_var = np.array([p.group == "var" for p in psms])
    pooled, sub = _pooled(psms), _subset_mask(psms)
    n_p, e_p = _fdp(w, psms, pooled & is_var)
    n_s, e_s = _fdp(w, psms, sub & is_var)
    n_r, e_r = _fdp(w, psms, pooled & ~is_var)
    print(f"world {world_id}: reference PSMs {n_r:>5} accepted, {e_r:>3} wrong ({100.0 * e_r / max(n_r, 1):4.1f}%) | variant PSMs "
          f"pooled {n_p:>3} accepted, {e_p:>3} wrong ({100.0 * e_p / max(n_p, 1):4.1f}%) | own estimate {n_s:>3} accepted, "
          f"{e_s:>3} wrong ({100.0 * e_s / max(n_s, 1):4.1f}%)")


def _compare_footer() -> None:
    rates = {"pooled": [], "own": []}
    for world_id in range(N_WORLDS):
        w = get_world(world_id)
        _, psms = searched(world_id, True)
        is_var = np.array([p.group == "var" for p in psms])
        for name, mask in (("pooled", _pooled(psms)), ("own", _subset_mask(psms))):
            n, e = _fdp(w, psms, mask & is_var)
            rates[name].append(100.0 * e / max(n, 1))
    print(f"wrong share among accepted variant PSMs, mean over {N_WORLDS} worlds ± standard error: "
          f"pooled {np.mean(rates['pooled']):.1f}% ± {_se(rates['pooled']):.1f}, own estimate {np.mean(rates['own']):.1f}% ± "
          f"{_se(rates['own']):.1f}; each was asked for {100.0 * ALPHA:g}%")
    own, pooled = float(np.mean(rates["own"])), float(np.mean(rates["pooled"]))
    print(f"the group's own estimate gives {'a lower' if own < pooled else 'no lower'} wrong share than the pooled one, and its "
          f"mean is {'above' if own > 100.0 * ALPHA else 'at or below'} the {100.0 * ALPHA:g}% it was asked for.")

if _DEMOS:
    _try_rows("section 7", "the pooled error rate against the variant group's own, one percent each:",
              [lambda w=w: _compare_row(w) for w in range(N_WORLDS)], _compare_footer, needs=("exercise 4", "exercise 5"))

## 8. Exercise 6 — `twin_verdict()`: could the spectrum be the reference peptide?

A variant peptide differs from its **reference twin** (the same peptide with the reference residue back in place) at one
residue. Most substitutions change the mass by a lot, so the twin is nowhere near the precursor and the search never
considers it. But the table in section 1 shows that lysine and glutamine differ by a few hundredths of a dalton, which is
inside the search window for a heavy peptide, and every fragment ion matches within the fragment tolerance either way. A
spectrum of the **reference** peptide can then be matched to the **variant** peptide with a very good score. Those wrong
matches are not chance matches, so the decoys, which count chance matches, cannot see them. How often the variant wins is
partly a matter of the engine's tie rule, which the table below counts: when both peptides explain the spectrum equally well,
this engine gives the spectrum to the alphabetically first one.

What can separate the two is **mass accuracy**: the spectrum's neutral mass is closer to the peptide it came from. Write
`twin_verdict(spectrum, peptide, offset, ref_aa)`. The twin is `peptide` with the residue at the 0-based position `offset`
replaced by `ref_aa`. Compare the two absolute differences, in daltons, exactly (no tolerance and no rounding), between the spectrum's neutral
mass (`neutral_mass`, given) and the mass of each peptide (`peptide_mass`, given). Return `"variant"` if the variant
peptide is strictly closer, `"twin"` if the twin is strictly closer and `"tie"` if the differences are equal (for example
when the twin differs only by leucine against isoleucine, or is the same peptide). Raise `ValueError` unless
`0 <= offset < len(peptide)`.

<details><summary>💡 Hint 1 — what to think about</summary>

What is the spectrum's mass, as a neutral mass and not an m/z? What does "closer" mean for a mass that can be above or below
the peptide's? What are the legal values of `offset`, and what does Python do with a negative index?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Check `offset` first. Build the twin by slicing around the position. Take the two absolute differences and compare them
with `<` in both directions, so that equality falls through to the third answer.
</details>

In [ ]:
def twin_verdict(spectrum: Spectrum, peptide: str, offset: int, ref_aa: str) -> str:
    """Which of a variant peptide and its reference twin has the mass closer to the spectrum's precursor.

    The twin is `peptide` with the residue at 0-based `offset` replaced by `ref_aa`. Compare `abs(neutral_mass(spectrum) -
    peptide_mass(p))` for `p` in the variant peptide and the twin, in daltons, exactly. Return `"variant"` if the variant is
    strictly closer, `"twin"` if the twin is, `"tie"` if the two are equal. Raise `ValueError` unless `0 <= offset <
    len(peptide)`.

    Example:
        >>> spec = Spectrum(0, (peptide_mass("AAAKAAAAK") + 0.01 + 2 * PROTON) / 2, 2, np.array([100.0]), np.array([1.0]))
        >>> twin_verdict(spec, "AAAKAAAAK", 3, "Q")      # the twin AAAQAAAAK is 0.036 Da lighter than the variant
        'variant'

    Returns:
        One of `"variant"`, `"twin"`, `"tie"`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _spectrum_at(mass: float, charge: int = 2) -> Spectrum:
    return Spectrum(0, mass / charge + PROTON, charge, np.array([100.0]), np.array([1.0]))


def _check_twin_verdict() -> None:
    pep = "AAAKAAAAK"                     # variant K at offset 3, reference residue Q
    heavy = peptide_mass(pep)
    light = peptide_mass("AAAQAAAAK")
    if not 0.03 < heavy - light < 0.04:
        raise RuntimeError("the lesson's own sanity check failed: K is a few hundredths of a dalton above Q")
    got = twin_verdict(_spectrum_at(heavy + 0.01), pep, 3, "Q")
    assert got == "variant", f"expected 'variant', got {got!r}: the spectrum is 0.01 Da from the variant and 0.046 Da from the twin"
    got = twin_verdict(_spectrum_at(light + 0.005), pep, 3, "Q")
    assert got == "twin", (
        f"expected 'twin', got {got!r}: the spectrum sits 0.005 Da above the twin. Only the residue at offset changes, so the twin "
        "of AAAKAAAAK at offset 3 keeps its other K; replace by position, not every copy of that residue")
    got = twin_verdict(_spectrum_at(light + 0.005, 3), pep, 3, "Q")
    assert got == "twin", (
        f"expected 'twin', got {got!r}: the spectrum (charge 3) sits 0.005 Da above the twin. Use neutral_mass, which uses the "
        "charge, not the precursor m/z")
    got = twin_verdict(_spectrum_at(light - 0.2), pep, 3, "Q")
    assert got == "twin", f"expected 'twin', got {got!r}: the spectrum is lighter than both, and the twin is the nearer one: compare absolute differences"
    got = twin_verdict(_spectrum_at(heavy + 0.2), pep, 3, "Q")
    assert got == "variant", f"expected 'variant', got {got!r}: the spectrum is heavier than both, and the variant is the nearer one"
    got = twin_verdict(_spectrum_at(peptide_mass("AAALAAK") + 0.003), "AAALAAK", 3, "I")
    assert got == "tie", f"expected 'tie', got {got!r}: L and I weigh exactly the same, so the two differences are equal"
    got = twin_verdict(_spectrum_at(heavy), pep, 3, "K")
    assert got == "tie", f"expected 'tie', got {got!r}: a twin that is the same peptide cannot be told apart from it"
    middle = (heavy + light) / 2
    above, below = twin_verdict(_spectrum_at(middle + 2e-7), pep, 3, "Q"), twin_verdict(_spectrum_at(middle - 2e-7), pep, 3, "Q")
    assert (above, below) == ("variant", "twin"), (
        f"expected ('variant', 'twin') for a spectrum 2e-7 Da either side of the midpoint, got ({above!r}, {below!r}): compare the two "
        "differences in daltons, strictly, with no tolerance and no ppm")
    pep2, ref2 = "AAASAAAAK", "AAAAAAAAK"           # serine against alanine, a pair whose half-gap does not sit on a rounding edge
    middle2 = (peptide_mass(pep2) + peptide_mass(ref2)) / 2
    for hair in (3e-7, 5e-11):
        above, below = twin_verdict(_spectrum_at(middle2 + hair), pep2, 3, "A"), twin_verdict(_spectrum_at(middle2 - hair), pep2, 3, "A")
        assert (above, below) == ("variant", "twin"), (
            f"expected ('variant', 'twin') for a spectrum {hair:g} Da either side of the midpoint, got ({above!r}, {below!r}): the two "
            "differences differ by less than a millionth of a dalton here, and the larger of them still loses; do not round them or "
            "treat a tiny difference as equal")
    first = _returned(lambda: twin_verdict(_spectrum_at(peptide_mass("WAAAAAK")), "WAAAAAK", 0, "G"),
                      "offset 0 is the first residue of a 7-residue peptide, a legal offset")
    last = _returned(lambda: twin_verdict(_spectrum_at(peptide_mass("AAAAAAW")), "AAAAAAW", 6, "G"),
                     "offset 6 is the last residue of a 7-residue peptide, a legal offset (0 <= offset < len(peptide))")
    assert first == "variant" and last == "variant", (
        f"expected 'variant' twice, got {first!r} and {last!r}: offset 0 is the first residue and len - 1 the last")
    for bad in (-1, 9, 10):
        assert _raised(ValueError, lambda: twin_verdict(_spectrum_at(heavy), pep, bad, "Q")) is not None, (
            f"offset {bad} is outside a peptide of 9 residues: ValueError expected (a negative one must not count from the end)")
    print("exercise 6 looks right: the closer of the variant and its twin is named, and ties are called ties")

In [ ]:
_try("exercise 6", _check_twin_verdict)

In [ ]:
def _peptide_variants(w: World) -> dict:
    """peptide -> tuple of the ids of the variants it is specific to, from the answer key. Given."""
    found: dict = {}
    for v, key in zip(w.variants, w.keys):
        for p, _ in key.peptides:
            found.setdefault(p, []).append(v.vid)
    return {p: tuple(ids) for p, ids in found.items()}


def _site_of(w: World) -> dict:
    """peptide -> (offset of the changed residue, the reference residue) for peptides that contain the change. Given."""
    site: dict = {}
    for v, key in zip(w.variants, w.keys):
        for p, offset in key.peptides:
            if offset is not None:
                site.setdefault(p, (offset, key.ref_aa))
    return site


def _after_twin(w: World, psms: list, mask) -> list:
    """The variant PSMs among `mask` that survive the twin check (YOUR `twin_verdict`). A peptide that does not contain the
    changed residue has no twin and is kept. Given."""
    site = _site_of(w)
    keep = []
    for p, m in zip(psms, mask):
        if not m or p.group != "var":
            continue
        if p.peptide in site:
            offset, ref_aa = site[p.peptide]
            if twin_verdict(w.spectra[p.scan], p.peptide, offset, ref_aa) != "variant":
                continue
        keep.append(p)
    return keep


def _twin_errors(w: World, space: dict, before: list) -> list:
    """The wrong PSMs among `before` whose spectrum was made from the reference twin of the peptide they were matched to, as
    `(psm, exact_tie, variant_residue, reference_residue)`. `exact_tie` is True when the twin, a target in the database, scores
    exactly the same hyperscore on that spectrum as the variant peptide did, so that only the engine's tie rule (a decoy first,
    then the alphabetically first peptide) chose the variant. Given."""
    site, found = _site_of(w), []
    for p in before:
        if not (_wrong(w, p) and p.peptide in site and w.source[p.scan] is not None):
            continue
        offset, ref_aa = site[p.peptide]
        if _norm(w.source[p.scan]) != _norm(p.peptide[:offset] + ref_aa + p.peptide[offset + 1:]):
            continue
        source = w.source[p.scan]
        tied = source in space and not space[source][0] and hyperscore(w.spectra[p.scan], source, FRAGMENT_TOL) == p.score
        found.append((p, tied, p.peptide[offset], ref_aa))
    return found


def _twin_row(world_id: int) -> None:
    w = get_world(world_id)
    space, psms = searched(world_id, True)
    mask = _subset_mask(psms) & np.array([p.group == "var" for p in psms])
    before = [p for p, m in zip(psms, mask) if m]
    after = _after_twin(w, psms, mask)
    errors = _twin_errors(w, space, before)
    e0, e1 = sum(_wrong(w, p) for p in before), sum(_wrong(w, p) for p in after)
    print(f"world {world_id}: own estimate {len(before):>3} accepted, {e0:>3} wrong ({len(errors):>2} of them the reference twin, "
          f"{sum(t for _, t, _, _ in errors):>2} of those an exact score tie) | "
          f"after the twin check {len(after):>3} accepted, {e1:>3} wrong ({100.0 * e1 / max(len(after), 1):4.1f}%)")


def _twin_footer() -> None:
    before_rates, after_rates, lost_right, lost_wrong = [], [], 0, 0
    example, errors = None, []
    for world_id in range(N_WORLDS):
        w = get_world(world_id)
        space, psms = searched(world_id, True)
        mask = _subset_mask(psms) & np.array([p.group == "var" for p in psms])
        before = [p for p, m in zip(psms, mask) if m]
        after = _after_twin(w, psms, mask)
        kept = {p.scan for p in after}
        errors += _twin_errors(w, space, before)
        for p in before:
            if p.scan not in kept:
                lost_wrong += _wrong(w, p)
                lost_right += not _wrong(w, p)
        for p, tied, variant_aa, ref_aa in _twin_errors(w, space, [p for p in before if p.scan not in kept]):
            if example is None:
                gap = (peptide_mass(p.peptide) - peptide_mass(w.source[p.scan])) / peptide_mass(p.peptide) * 1e6
                example = (w.source[p.scan], p.peptide, gap, tied)
        before_rates.append(100.0 * sum(_wrong(w, p) for p in before) / max(len(before), 1))
        after_rates.append(100.0 * sum(_wrong(w, p) for p in after) / max(len(after), 1))
    print(f"wrong share among accepted variant PSMs, mean ± standard error over {N_WORLDS} worlds: "
          f"own estimate {np.mean(before_rates):.1f}% ± {_se(before_rates):.1f}, after the twin check "
          f"{np.mean(after_rates):.1f}% ± {_se(after_rates):.1f}")
    print(f"the check removed {lost_wrong} wrong and {lost_right} right variant PSMs")
    if example:
        print(f"example of a wrong match the check removed: a spectrum of {example[0]} matched to {example[1]}, "
              f"{example[2]:.1f} ppm apart in mass, " + ("with exactly the same score for both" if example[3] else "the variant scoring higher"))
    tied = [e for e in errors if e[1]]
    pairs = sorted({f"{v} in place of {r}" for _, _, v, r in tied})
    print(f"of the {len(errors)} accepted variant PSMs whose spectrum was made from the reference twin, {len(tied)} had exactly the "
          f"same score for the twin and the variant (variant residue against reference residue: {', '.join(pairs) or 'none'}), and "
          f"were given to the variant by this engine's tie rule, the alphabetically first peptide; an engine that reported such a "
          f"tie as ambiguous, or kept the reference, would not make those {len(tied)} errors, so how many the twin check has to "
          f"remove depends on that rule")


if _DEMOS:
    _try_rows("section 8", "the variant group's own estimate, then the twin check:",
              [lambda w=w: _twin_row(w) for w in range(N_WORLDS)], _twin_footer,
              needs=("exercise 4", "exercise 5", "exercise 6"))

## 9. Exercise 7 — `variant_support()`: how many peptides stand behind each variant

A variant is evidence at the level of **protein** only through the peptides that carry it. One spectrum is a weak witness:
the same peptide can be seen in many spectra, and a repeated peptide is still one peptide. The usual reading is by the number
of **distinct peptide sequences** accepted for the variant: two or more is **multi**-peptide support, exactly one is
**single**-peptide support, and none is no support.

Write `variant_support(peptides, peptide_variants, variant_ids)`. `peptides` is an iterable of accepted peptide strings, one
per accepted PSM, so a peptide seen in three spectra appears three times (it may be a generator). `peptide_variants` maps a
variant peptide to the tuple of ids of the variants it is specific to; a peptide that is not in it (a reference peptide) is
ignored, and a peptide specific to two variants counts for both. Return a dict with one entry for every id of `variant_ids`,
in the order given (an id that is given twice is one entry), whose value is `"multi"`, `"single"` or `"none"`. Count distinct
peptide strings only. Leave the arguments unchanged; `variant_ids` may be any sequence of ids.

<details><summary>💡 Hint 1 — what to think about</summary>

What do you collect for each variant: spectra, peptides or distinct peptides? Which ids belong in the answer, and what
should a peptide that no variant claims do?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Keep one set of peptides for each requested id. Walk through the accepted peptides once, and add each to the set of every
variant it is specific to. Turn the size of each set into one of the three words.
</details>

In [ ]:
def variant_support(peptides, peptide_variants: dict, variant_ids: list) -> dict:
    """Spectral support class of each variant, by the number of distinct accepted peptides specific to it.

    Count, for each id in `variant_ids`, the distinct peptide strings in `peptides` whose entry in `peptide_variants`
    lists that id (a peptide missing from the dict is ignored; one listed for two ids counts for both). Two or more is
    `"multi"`, one is `"single"`, none is `"none"`. Return a dict with exactly the ids of `variant_ids`, in that order (an id given
    twice is one entry).

    Example:
        >>> variant_support(["P1", "P1", "P2", "R"], {"P1": (7,), "P2": (7, 8)}, [7, 8, 9])
        {7: 'multi', 8: 'single', 9: 'none'}

    Returns:
        A dict from variant id to one of the three words.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_variant_support() -> None:
    got = variant_support(["P1", "P1", "P2", "R"], {"P1": (7,), "P2": (7, 8)}, [7, 8, 9])
    assert got == {7: "multi", 8: "single", 9: "none"} and list(got) == [7, 8, 9], (
        f"expected {{7: 'multi', 8: 'single', 9: 'none'}} in that order, got {got}. Variant 7 has two distinct peptides; "
        "variant 8 has one, shared with 7; variant 9 has none")
    got = variant_support(["P1", "P2", "P3"], {"P1": (3,), "P2": (3,), "P3": (3,)}, [3])
    assert got == {3: "multi"}, (
        f"expected {{3: 'multi'}}, got {got}: two OR MORE distinct peptides is multi, so three is multi too, not only exactly two")
    got = variant_support(["P1", "P1", "P1"], {"P1": (3,)}, [3])
    assert got == {3: "single"}, (
        f"expected {{3: 'single'}}, got {got}: one peptide seen in three spectra is single-peptide support, not multi")
    got = variant_support(iter(["P2", "P1"]), {"P1": (3, 4), "P2": (3,)}, [3, 4])
    assert got == {3: "multi", 4: "single"}, f"expected {{3: 'multi', 4: 'single'}}, got {got}: the peptides may come from a generator, read once for all the ids"
    got = variant_support(["R1", "R2"], {"P1": (3,)}, [3, 4])
    assert got == {3: "none", 4: "none"}, f"expected both 'none', got {got}: reference peptides are not evidence for a variant"
    got = variant_support(["P1", "P5"], {"P1": (3,), "P5": (5,)}, [3])
    assert got == {3: "single"}, f"expected {{3: 'single'}}, got {got}: only the ids asked for are reported"
    got = variant_support(["P1"], {"P1": (3, 4)}, [4, 3])
    assert got == {4: "single", 3: "single"} and list(got) == [4, 3], (
        f"got {got}: a peptide specific to two variants counts for both, and the order of variant_ids is kept")
    assert variant_support([], {}, []) == {}, "no ids, no answer"
    got = variant_support(["P1"], {"P1": ()}, [3])
    assert got == {3: "none"}, f"expected {{3: 'none'}}, got {got}: a peptide whose entry lists no variant supports none"
    for a, b in (("I", "L"), ("L", "I"), ("K", "Q"), ("E", "D")):
        two = ["AAA" + a + "AAAK", "AAA" + b + "AAAK"]
        got = variant_support(two, {two[0]: (3,), two[1]: (3,)}, [3])
        assert got == {3: "multi"}, (
            f"expected {{3: 'multi'}}, got {got}: {two[0]} and {two[1]} are two different peptide strings. Count distinct strings as they "
            "are; reading I as L, or any residue as another, belongs to novel_peptides and not to this count")
    two = ["AAAAAAK", "aaaaaak"]
    got = variant_support(two, {two[0]: (3,), two[1]: (3,)}, [3])
    assert got == {3: "multi"}, f"expected {{3: 'multi'}}, got {got}: count distinct peptide strings as they are, upper and lower case included"
    got = variant_support(["P1"], {"P1": (3,)}, [3, 3])
    assert got == {3: "single"}, f"expected {{3: 'single'}}, got {got}: an id that is given twice is one entry"
    ids = [4, 3]
    variant_support(["P1"], {"P1": (3,)}, ids)
    assert ids == [4, 3], "do not change variant_ids (do not sort it or empty it)"
    table = {"P1": (3,), "P2": (3,)}
    keep = dict(table)
    got = variant_support(["P1", "P2"], table, (3, 9))
    assert got == {3: "multi", 9: "none"} and table == keep, f"got {got}: variant_ids may be a tuple, and peptide_variants is left unchanged"
    table = {"P1": (3,), "P2": (3,)}
    keep = dict(table)
    got = variant_support(["P1", "R1"], table, [3])
    assert got == {3: "single"} and table == keep and list(table) == ["P1", "P2"], (
        f"got {got}, and peptide_variants is now {table}: do not change peptide_variants, not even to add an entry for a peptide "
        "that is not in it (a reference peptide is simply ignored)")
    print("exercise 7 looks right: distinct peptides per variant, three classes")

In [ ]:
_try("exercise 7", _check_variant_support)

In [ ]:
def _regimes(world_id: int) -> dict:
    """Accepted variant peptides (one per PSM) under each way of controlling the error. Given; uses YOUR exercises 4 to 6."""
    w = get_world(world_id)
    _, psms = searched(world_id, True)
    is_var = np.array([p.group == "var" for p in psms])
    own = _subset_mask(psms) & is_var
    return {"pooled": [p.peptide for p, m in zip(psms, _pooled(psms) & is_var) if m],
            "own estimate": [p.peptide for p, m in zip(psms, own) if m],
            "own estimate and twin check": [p.peptide for p in _after_twin(w, psms, own)]}


def _support_counts(world_id: int) -> dict:
    w = get_world(world_id)
    ids = [v.vid for v in w.variants]
    table = {}
    for name, peps in _regimes(world_id).items():
        classes = variant_support(peps, _peptide_variants(w), ids)
        rows = {"multi": [0, 0], "single": [0, 0]}
        for vid, c in classes.items():
            if c != "none":
                rows[c][0] += 1
                rows[c][1] += vid not in w.observable
        table[name] = rows
    return table


def _support_row(world_id: int) -> None:
    counts = _support_counts(world_id)
    w = get_world(world_id)
    text = " | ".join(f"{name}: {r['multi'][0] + r['single'][0]:>3} reported, {r['multi'][1] + r['single'][1]:>3} unsupported"
                      for name, r in counts.items())
    print(f"world {world_id} ({len(w.observable)} variants really seen): {text}")


def _support_footer() -> None:
    names = ("pooled", "own estimate", "own estimate and twin check")
    total = {n: {"multi": [0, 0], "single": [0, 0]} for n in names}
    seen = 0
    for world_id in range(N_WORLDS):
        seen += len(get_world(world_id).observable)
        for name, rows in _support_counts(world_id).items():
            for c in ("multi", "single"):
                for k in (0, 1):
                    total[name][c][k] += rows[c][k]
    print(f"\nover {N_WORLDS} worlds ({seen} variants have a spectrum made from one of their peptides):")
    print(f"  {'regime':<30} {'multi':>14} {'single':>14}   (reported, of which no spectrum was made from the variant)")
    for name in names:
        m, s = total[name]["multi"], total[name]["single"]
        print(f"  {name:<30} {m[0]:>5}, {m[1]:>3} none {s[0]:>5}, {s[1]:>3} none")
    last = total[names[-1]]
    single_rate = 100.0 * last["single"][1] / max(last["single"][0], 1)
    multi_rate = 100.0 * last["multi"][1] / max(last["multi"][0], 1)
    reported = last["multi"][0] + last["single"][0]
    unseen = last["multi"][1] + last["single"][1]
    print(f"under the strictest regime {single_rate:.1f}% of the single-peptide variants and {multi_rate:.1f}% of the "
          f"multi-peptide variants have no spectrum behind them; it reports {reported} variants, {reported - unseen} of the "
          f"{seen} that could be seen and {unseen} that could not")


if _DEMOS:
    _try_rows("section 9", "variants reported by each regime, with the answer key's verdict:",
              [lambda w=w: _support_row(w) for w in range(N_WORLDS)], _support_footer,
              needs=("exercise 4", "exercise 5", "exercise 6", "exercise 7"))

## 10. Common mistakes

- **Reading the strand wrongly.** On the minus strand the coding sequence starts at `end - 1` and carries the complement of
  the base a variant file gives. A variant called against the wrong strand is a different variant.
- **Treating every variant as a candidate peptide.** Synonymous variants, stop-loss variants in this lesson, and a swap of
  leucine for isoleucine leave nothing a search can find. Counting them as "variants without support" blames the data for
  what the biology and this engine never allowed.
- **Comparing a variant peptide with one gene's peptides.** The reference is the whole proteome: a peptide another protein
  already makes is not specific to the variant.
- **Giving a variant peptide's decoy the reference group.** Then the variant group has no decoys of its own and its error
  rate cannot be estimated. Keep the group of the target the decoy came from.
- **Trusting a pooled one percent for a small group.** The pooled rate averages over a crowd that is mostly true hits. The
  group's own rate is only an estimate too: with few decoys it is noisy, and a group with almost none cannot be estimated at all.
- **Taking a clean decoy count for a clean list.** Decoys count chance matches. A spectrum of one peptide matched to a
  near-identical peptide is not chance, so no decoy count sees it; check the mass.
- **Counting spectra as evidence.** Five spectra of one peptide are one peptide.
- **Reading these numbers as a rate for real data.** They describe these worlds, one engine and one kind of near-isobaric pair.
What this does not cover: introns and splice variants, haplotypes, insertions and deletions, modified peptides, transcript-level
evidence, and the fact that two missed-cleavage peptides around one change are counted as two peptides although they are one
piece of evidence about one site.

## 11. Self-check

1. A search over a database of reference and variant peptides is run at a pooled error rate of one percent. What does that
   tell you about the PSMs for variant peptides?
   - (a) about one percent of them are wrong, like every other group of PSMs
   - (b) nothing specific: the rate is an average over all PSMs, and a group made mostly of peptides that are not in the sample can
     have many more wrong hits per accepted hit
   - (c) fewer than one percent are wrong, because variant peptides are checked against the reference as well
2. Why do reversed decoys not reveal a spectrum of the reference peptide that was matched to its variant twin, when the two
   differ by glutamine against lysine?
   - (a) decoys count chance matches, and this match is not chance: the variant peptide explains the spectrum about as well as the true peptide does
   - (b) decoys are made from the reference only, so none of them is near a variant peptide
   - (c) the decoy of a variant peptide is always dropped because it equals a target
3. A variant has one tryptic peptide, accepted in three different spectra. How much support is that?
   - (a) multi-peptide: three spectra are three witnesses
   - (b) none, because a variant needs at least two spectra from different charge states
   - (c) single-peptide: one peptide, seen three times
4. Which of these variants can never be supported by a spectrum in this lesson, however many spectra you collect?
   - (a) a synonymous variant, and a variant that swaps leucine for isoleucine
   - (b) a missense variant in a low-abundance protein
   - (c) a variant on the minus strand

Answers come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _state(label: str) -> str:
    """An exercise is failed when its own check failed, or when a table row below it disagreed with the answer key."""
    state = _STATUS.get(label, "not started")
    return "failed" if state == "passed" and _ROW_FAILS.get(label) else state


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check and of the table rows that test it, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _state(label)
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_state(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _state(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
        rows = [f"{label}: {'; '.join(sorted(_ROW_FAILS[label]))}" for label in _EXERCISES if _ROW_FAILS.get(label)]
        if rows:
            print("a table below an exercise disagreed with the answer key (" + "; ".join(rows) + "): fix the function, "
                  "then re-run that table's cell, because the board reads its latest run.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (  # (exercise, its check)
        ("exercise 1", _check_translate),
        ("exercise 2", _check_snv_effect),
        ("exercise 3", _check_novel_peptides),
        ("exercise 4", _check_build_space),
        ("exercise 5", _check_group_q_values),
        ("exercise 6", _check_twin_verdict),
        ("exercise 7", _check_variant_support),
    )
    import contextlib
    import io
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in _ALL_CHECKS:
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends the run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

## What you built, and what it cannot see

You built the path from a called variant to a claim about protein: the effect of a variant on a gene on either strand, the
peptides that exist only because of it (with leucine and isoleucine read as one), a database whose decoys keep their group, an
error rate for the variant group alone, a mass check against the reference twin, and a count of distinct peptides per
variant. **What it cannot see:** a wrong match that is close to the truth in every way the engine looks at, such as an
isobaric pair that the mass accuracy cannot separate; variants that need introns, haplotypes or modifications to be
understood; and anything about a real cohort, whose reference database, variant calls and spectra are messier than these.
A variant that survives every step has, at best, spectral evidence of a peptide. Whether the person carries the protein is
another question. And some of the errors the twin check removed exist only because of this engine's tie rule: where the twin and the
variant score exactly the same, the alphabetically first peptide wins, and the table in section 8 counted how many of the errors
were such ties. An engine that handled ties differently would show a different number.

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.